In [6]:
import os
import sys
import json
import re
import hashlib
import logging
from pathlib import Path
from pprint import pprint
from dotenv import load_dotenv
from openai import AzureOpenAI
from neo4j import GraphDatabase
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential

repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / 'ai-services').exists() and (candidate / '.env').exists():
        repo_root = candidate
        break

ai_services_dir = repo_root / 'ai-services'
if str(ai_services_dir) not in sys.path:
    sys.path.insert(0, str(ai_services_dir))

load_dotenv(repo_root / '.env', override=True)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger('semantic_kg_debug')

print('repo_root =', repo_root)
print('ai_services_dir =', ai_services_dir)


repo_root = /Users/senghok/Projects/gds_tender_test/output/jupyter-notebook
ai_services_dir = /Users/senghok/Projects/gds_tender_test/output/jupyter-notebook/ai-services


In [2]:
# from src.services.APIM_connection import ApimSubscriptionPolicy
# from src.document_processing.utils import extract_page_and_chunk, escape_odata_string

In [7]:
import re

def extract_page_and_chunk(doc_id: str):
    """
    Returns a sortable tuple.
    - PDF / Word: (page_number, chunk_number)
    - Excel:      (sheet_name, chunk_number)

    Expected ID formats:
      *_page_12_text_chunk_3  (for PDFs with text chunks)
      *_page_12_chunk_3       (alternative format)
      *_page_Sheet1_chunk_7   (for Excel sheets)
    """
    # Extract chunk number (at the end of the string)
    chunk_match = re.search(r"_chunk_(\d+)$", doc_id)
    chunk = int(chunk_match.group(1)) if chunk_match else 0

    # Extract page or sheet name
    # Look for _page_XXX_ where XXX can be:
    #  - digits (page number)
    #  - sheet name followed by _chunk_
    # Use non-greedy match and stop at _text_ or _chunk_
    page_match = re.search(r"_page_([^_]+?)(?:_text)?_chunk_", doc_id)
    page = page_match.group(1) if page_match else ""

    # Numeric page → PDF / Word
    if page.isdigit():
        return (int(page), chunk)
    # Sheet name → Excel (case-insensitive sort)
    return (page.lower(), chunk)

def escape_odata_string(value: str) -> str:
    """
    Escapes single quotes for OData filters.
    Example: O'Brien → O''Brien
    """
    return value.replace("'", "''")


In [8]:

from azure.core.pipeline.policies import SansIOHTTPPolicy
from azure.core.credentials import AccessToken
import re
import logging
import os

logger = logging.getLogger(__name__)
API_KEY=os.getenv("API_KEY")

class ApimSubscriptionPolicy(SansIOHTTPPolicy):
    """Custom policy for APIM Subscription Key Authentication"""
    def __init__(self, subscription_key: str):
        super().__init__()
        self.subscription_key = subscription_key.strip() if subscription_key else ""
        self.operation_id = None 

    def on_request(self, request):
        """Add Ocp-Apim-Subscription-Key header to all requests"""
        if not self.subscription_key:
            logger.warning("APIM subscription key is empty")
            return
    
        if "authorization" in request.http_request.headers:
            del request.http_request.headers["authorization"]
        
        request.http_request.headers["Ocp-Apim-Subscription-Key"] = self.subscription_key
        logger.debug(f"Added APIM header to request: {request.http_request.method} {request.http_request.url[:100]}")
        
    def on_response(self, request, response):
        """Extract and store operation ID from Operation-Location header"""
        if response.http_response.status_code >= 400:
            logger.error(f"APIM Response Error: {response.http_response.status_code}")
            if hasattr(response.http_response, 'text'):
                response_text = response.http_response.text() if callable(response.http_response.text) else response.http_response.text
                logger.error(f"Response body: {response_text[:500]}") if response_text else None
        
        if "Operation-Location" in response.http_response.headers:
            operation_location = response.http_response.headers["Operation-Location"]
            match = re.search(r'/analyzeResults/([a-f0-9\-]{36})', operation_location)
            if match:
                self.operation_id = match.group(1)
                logger.debug(f"Extracted operation ID: {self.operation_id}")

class NoOpCredential:
    """Dummy credential for APIM-based authentication"""
    def get_token(self, *args, **kwargs):
        return AccessToken(API_KEY, 0)


In [9]:

TARGET_ITB_ID = 'ROC_INPEX'
WRITE_TO_NEO4J = True
RESET_SEMANTIC_NODES_FOR_DOC = False

env_check = {
    'AZURE_OPENAI_APIM': os.getenv('AZURE_OPENAI_APIM'),
    'APIM_SUBSCRIPTION_KEY': os.getenv('APIM_SUBSCRIPTION_KEY'),
    'AZURE_OPENAI_API_VERSION': os.getenv('AZURE_OPENAI_API_VERSION'),
    'AI_SEARCH_APIM': os.getenv('AI_SEARCH_APIM'),
    'AZURE_SEARCH_INDEX_NAME': os.getenv('AZURE_SEARCH_INDEX_NAME'),
    'NEO4J_URI': os.getenv('NEO4J_URI'),
    'NEO4J_USER': os.getenv('NEO4J_USER'),
}
pprint(env_check)

openai_client = AzureOpenAI(
    azure_endpoint=os.getenv('AZURE_OPENAI_APIM'),
    api_key='DUMMY',
    api_version=os.getenv('AZURE_OPENAI_API_VERSION', '2024-02-15-preview'),
    default_headers={'Ocp-Apim-Subscription-Key': os.getenv('APIM_SUBSCRIPTION_KEY')},
)
 
search_client = SearchClient(
    endpoint=os.getenv('AI_SEARCH_APIM'),
    index_name='llm4t-documents-index-hok',
    credential=AzureKeyCredential('DUMMY'),
    per_call_policies=[ApimSubscriptionPolicy(os.getenv("APIM_SUBSCRIPTION_KEY"))],
)

neo4j_driver = GraphDatabase.driver(
    os.getenv('NEO4J_URI'),
    auth=(os.getenv('NEO4J_USER'), os.getenv('NEO4J_PASSWORD')),
    connection_timeout=10,
    max_connection_lifetime=600,
)

chat_deployment = (
    os.getenv('AZURE_OPENAI_VISION_DEPLOYMENT')
)

print('chat_deployment =', chat_deployment)


{'AI_SEARCH_APIM': 'https://apis-dev.lab.technipenergies.com/sales/llmfortendering/aisearch/',
 'APIM_SUBSCRIPTION_KEY': '25ac70c2fcdd43d789c079f7fc91985c',
 'AZURE_OPENAI_APIM': 'https://apis-dev.lab.technipenergies.com/sales/llmfortendering/cognitiveservices/',
 'AZURE_OPENAI_API_VERSION': '2024-12-01-preview',
 'AZURE_SEARCH_INDEX_NAME': 'llm4t-documents-index-hok',
 'NEO4J_URI': 'neo4j://127.0.0.1:7687',
 'NEO4J_USER': 'neo4j'}
chat_deployment = gpt-4o-llm4t


In [10]:
search_client = SearchClient(
    endpoint=os.getenv('AI_SEARCH_APIM'),
    index_name='llm4t-documents-index-hok',
    credential=AzureKeyCredential('DUMMY'),
    per_call_policies=[ApimSubscriptionPolicy(os.getenv("APIM_SUBSCRIPTION_KEY"))],
)

### Delete from Neo4j

In [11]:
# deleted_file_names = [
#     "ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 03_Exhibit A13_Contractor Documentation_Rev.1_20250808_Track Change.pdf",
#     "ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 03_Exhibit A7_Construction_Rev2.20250808.Track Change.pdf",
#     "ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 03_Exhibit A3_Engineering_Rev.2_20250808_Change.pdf",
#     "ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 04_Exhibit B_Compensation_Rev.3_20250808_Track Change.pdf",
#     "ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 03_Exhibit A10_Project Interface Plans_Rev.1_20250808_Track Change_.pdf",
#     "ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 08_Exhibit F_Health Safety Environment and Security (HSES)_Rev.1_20250808_Track Change.pdf",
#     "ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 03_Exhibit A8_Completions Pre-Commissioning Commissioning and Handover_Rev.2_20250808_Track Change.pdf",
#     "ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 03_Exhibit A1_Site Maps_Rev.1_20250808_Track Change_.pdf",
#     "ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 03_Exhibit A_Scope of Work_Rev.2_20250808_Track Change.pdf",
#     "ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 03_Exhibit A5_Modul Yard_Rev.2_20250808_Track Change.pdf",
#     "ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 03_Exhibit A12_Environmental Conditions_Rev.1_20250808_Track Change.pdf",
#     "ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 05_Exhibit C_Work Time Schedule_Rev.3_20250808_Track Change.pdf",
#     "ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 07_Exhibit E_Design Dossier_Rev.2_20250808_Track Change.pdf",
#     "ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 03_Exhibit A9_Company Operability Assurance_Rev.1_20250808_Track Change.pdf",
#     "ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 03_Exhibit A11_Contractor Provided Services_Rev.2_20250808_Track Change.pdf",
# ]

# deleted_file_basenames = [name.split("/")[-1] for name in deleted_file_names]
# dry_run_cypher = """
# MATCH (d:Document)
# WHERE d.docURL IN $deleted_file_names
#    OR d.fileName IN $deleted_file_names
#    OR d.docURL IN $deleted_file_basenames
#    OR d.fileName IN $deleted_file_basenames
# WITH collect(DISTINCT d) AS docs

# OPTIONAL MATCH (ch:Chunk)-[:PART_OF]->(d)
# WHERE d IN docs
# WITH docs, collect(DISTINCT ch) AS chunks

# OPTIONAL MATCH (ch)-[:ASSERTS]->(r:Relation)
# WHERE ch IN chunks
# WITH docs, chunks, collect(DISTINCT r) AS relations

# RETURN
#   size(docs) AS documents_to_delete,
#   size(chunks) AS chunks_to_delete,
#   size(relations) AS relations_to_delete,
#   [d IN docs | coalesce(d.docURL, d.fileName)] AS matched_documents
# """

# with neo4j_driver.session() as s:
#     dry_run_rows = [
#         r.data()
#         for r in s.run(
#             dry_run_cypher,
#             {
#                 "deleted_file_names": deleted_file_names,
#                 "deleted_file_basenames": deleted_file_basenames,
#             },
#         )
#     ]

# delete_cypher = """
# MATCH (d:Document)
# WHERE d.docURL IN $deleted_file_names
#    OR d.fileName IN $deleted_file_names
#    OR d.docURL IN $deleted_file_basenames
#    OR d.fileName IN $deleted_file_basenames
# WITH collect(DISTINCT d) AS docs

# OPTIONAL MATCH (ch:Chunk)-[:PART_OF]->(d)
# WHERE d IN docs
# WITH docs, collect(DISTINCT ch) AS chunks

# WITH docs, chunks,
#      size(docs) AS document_count,
#      size(chunks) AS chunk_count

# FOREACH (ch IN chunks | DETACH DELETE ch)
# FOREACH (d IN docs | DETACH DELETE d)

# RETURN
#   document_count AS deleted_documents,
#   chunk_count AS deleted_chunks
# """

# with neo4j_driver.session() as s:
#     delete_rows = [
#         r.data()
#         for r in s.run(
#             delete_cypher,
#             {
#                 "deleted_file_names": deleted_file_names,
#                 "deleted_file_basenames": deleted_file_basenames,
#             },
#         )
#     ]

# delete_rows

### Delete files whose name contain "Change"

In [12]:
# from azure.core.exceptions import HttpResponseError

# # 1. Find matching file names from the facet list
# results = search_client.search(
#     search_text="*",
#     facets=["file_name,count:200"],
#     top=0,
# )

# all_file_names = [f["value"] for f in results.get_facets().get("file_name", [])]
# change_file_names = [name for name in all_file_names if "Change" in name]

# print(f"Total files in index: {len(all_file_names)}")
# print(f"Files matching 'Change': {len(change_file_names)}")
# for name in change_file_names:
#     print(name)

# from azure.core.exceptions import HttpResponseError

# # Find file names containing "Change"
# results = search_client.search(
#     search_text="*",
#     facets=["file_name,count:200"],
#     top=0,
# )

# all_file_names = [f["value"] for f in results.get_facets().get("file_name", [])]
# change_file_names = [name for name in all_file_names if "Change" in name]

# print(f"Total files in index: {len(all_file_names)}")
# print(f"Files matching 'Change': {len(change_file_names)}")
# for name in change_file_names:
#     print(name)

# # Fetch matching chunk ids
# docs_to_delete = []

# for file_name in change_file_names:
#     escaped_file_name = file_name.replace("'", "''")

#     docs = search_client.search(
#         search_text="*",
#         filter=f"file_name eq '{escaped_file_name}'",
#         select=["chunk_id", "file_name"],
#         top=1000,
#     )

#     for doc in docs:
#         docs_to_delete.append({
#             "chunk_id": doc["chunk_id"],
#         })

# print(f"Chunks to delete: {len(docs_to_delete)}")

# # Delete chunks by key field
# batch_size = 1000

# for i in range(0, len(docs_to_delete), batch_size):
#     batch = docs_to_delete[i : i + batch_size]

#     try:
#         result = search_client.delete_documents(documents=batch)
#         print(f"Deleted batch {i // batch_size + 1}: {len(batch)} chunks")
#     except HttpResponseError as e:
#         print(f"Failed batch {i // batch_size + 1}: {e}")

# print("Done.")

In [13]:
results = search_client.search(
    search_text="*",
    facets=["file_name,count:200"],
    top=0
)
all_file_names = [f["value"] for f in results.get_facets().get("file_name", [])]
print(f"Total files in index: {len(all_file_names)}")

# Dict[file_name, List[chunk]]
all_files_docs = {}
batch_size = 1000

for file_name in all_file_names:
    docs = []
    skip = 0

    while True:
        results = search_client.search(
            search_text='*',
            filter=f"file_name eq '{escape_odata_string(file_name)}'",
            select=[
                'chunk_id', 'tender_id', 'file_name', 'page_chunk', 'blob_url',
                'chunk_index', 'page_number', 'chunk_type', 'is_active',
                'high_level_domain', 'mid_level_domain', 'low_level_domain'
            ],
            top=batch_size,
            skip=skip,
        )
        batch = list(results)
        if not batch:
            break
        docs.extend(batch)
        if len(batch) < batch_size:
            break
        skip += batch_size
        if skip >= 100000:
            break

    all_files_docs[file_name] = sorted(docs, key=lambda d: extract_page_and_chunk(d['chunk_id']))
    print(f"  [{len(all_files_docs)}/{len(all_file_names)}] {file_name}: {len(docs)} chunks")

print(f"\nDone. {len(all_files_docs)} files, {sum(len(v) for v in all_files_docs.values())} total chunks")


2026-07-19 16:58:56,298 - INFO - Request URL: 'https://apis-dev.lab.technipenergies.com/sales/llmfortendering/aisearch//indexes('llm4t-documents-index-hok')/docs/search.post.search?api-version=REDACTED'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '60'
    'api-key': 'REDACTED'
    'Accept': 'application/json;odata.metadata=none'
    'Ocp-Apim-Subscription-Key': 'REDACTED'
    'x-ms-client-request-id': '5d820c02-8382-11f1-a221-46d9942f8b90'
    'User-Agent': 'azsdk-python-search-documents/11.6.0 Python/3.13.5 (macOS-14.3-arm64-arm-64bit-Mach-O)'
A body is sent with the request
2026-07-19 16:58:56,554 - INFO - Response status: 200
Response headers:
    'content-type': 'application/json; odata.metadata=none; odata.streaming=true; charset=utf-8'
    'content-encoding': 'REDACTED'
    'vary': 'REDACTED'
    'preference-applied': 'REDACTED'
    'odata-version': 'REDACTED'
    'request-id': '5d820c02-8382-11f1-a221-46d9942f8b90'
    'el

Total files in index: 84


2026-07-19 16:58:57,350 - INFO - Response status: 200
Response headers:
    'content-type': 'application/json; odata.metadata=none; odata.streaming=true; charset=utf-8'
    'content-encoding': 'REDACTED'
    'vary': 'REDACTED'
    'preference-applied': 'REDACTED'
    'odata-version': 'REDACTED'
    'request-id': '5da9722e-8382-11f1-a221-46d9942f8b90'
    'elapsed-time': 'REDACTED'
    'request-context': 'REDACTED'
    'date': 'Sun, 19 Jul 2026 14:58:56 GMT'
    'x-envoy-upstream-service-time': 'REDACTED'
    'strict-transport-security': 'REDACTED'
    'x-volterra-location': 'REDACTED'
    'server': 'web'
    'transfer-encoding': 'chunked'
2026-07-19 16:58:57,382 - INFO - Request URL: 'https://apis-dev.lab.technipenergies.com/sales/llmfortendering/aisearch//indexes('llm4t-documents-index-hok')/docs/search.post.search?api-version=REDACTED'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '337'
    'api-key': 'REDACTED'
    'Accept': 'ap

  [1/84] ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 03_Exhibit A_Scope of Work_Rev.0_20241216.pdf: 741 chunks


2026-07-19 16:58:58,184 - INFO - Response status: 200
Response headers:
    'content-type': 'application/json; odata.metadata=none; odata.streaming=true; charset=utf-8'
    'content-encoding': 'REDACTED'
    'vary': 'REDACTED'
    'preference-applied': 'REDACTED'
    'odata-version': 'REDACTED'
    'request-id': '5e276472-8382-11f1-a221-46d9942f8b90'
    'elapsed-time': 'REDACTED'
    'request-context': 'REDACTED'
    'date': 'Sun, 19 Jul 2026 14:58:57 GMT'
    'x-envoy-upstream-service-time': 'REDACTED'
    'strict-transport-security': 'REDACTED'
    'x-volterra-location': 'REDACTED'
    'server': 'web'
    'transfer-encoding': 'chunked'
2026-07-19 16:58:58,207 - INFO - Request URL: 'https://apis-dev.lab.technipenergies.com/sales/llmfortendering/aisearch//indexes('llm4t-documents-index-hok')/docs/search.post.search?api-version=REDACTED'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '360'
    'api-key': 'REDACTED'
    'Accept': 'ap

...

  [77/84] ROC_INPEX/Inpex AGRU CCS _FEED validation contract_Received via Ariba/Annexure B - Form of Tender.docx: 2 chunks
  [78/84] ROC_INPEX/Inpex AGRU CCS _FEED validation contract_Received via Ariba/ITT Part B - Information Requested by Company_Technical.docx: 2 chunks
  [79/84] ROC_INPEX/Inpex AGRU CCS _FEED validation contract_Received via Ariba/Template - Log of Tenderers Commercial Qualifications Rev0.docx: 2 chunks


2026-07-19 16:59:15,922 - INFO - Response status: 200
Response headers:
    'content-type': 'application/json; odata.metadata=none; odata.streaming=true; charset=utf-8'
    'content-encoding': 'REDACTED'
    'vary': 'REDACTED'
    'preference-applied': 'REDACTED'
    'odata-version': 'REDACTED'
    'request-id': '69284f3a-8382-11f1-a221-46d9942f8b90'
    'elapsed-time': 'REDACTED'
    'request-context': 'REDACTED'
    'date': 'Sun, 19 Jul 2026 14:59:15 GMT'
    'x-envoy-upstream-service-time': 'REDACTED'
    'strict-transport-security': 'REDACTED'
    'x-volterra-location': 'REDACTED'
    'server': 'web'
    'transfer-encoding': 'chunked'
2026-07-19 16:59:15,924 - INFO - Request URL: 'https://apis-dev.lab.technipenergies.com/sales/llmfortendering/aisearch//indexes('llm4t-documents-index-hok')/docs/search.post.search?api-version=REDACTED'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '359'
    'api-key': 'REDACTED'
    'Accept': 'ap

  [80/84] ROC_INPEX/Inpex AGRU CCS _FEED validation contract_Received via Ariba/Template - Log of Tenderers Contractual Qualifications Rev0.docx: 2 chunks
  [81/84] ROC_INPEX/Inpex AGRU CCS _FEED validation contract_Received via Ariba/Template - Log of Tenderers Technical Qualifications Rev0.docx: 2 chunks
  [82/84] ROC_INPEX/Package A Submission/Package A (Contractual Evaluation Pack) Submission Requirement.pdf: 2 chunks


2026-07-19 16:59:16,185 - INFO - Response status: 200
Response headers:
    'content-type': 'application/json; odata.metadata=none; odata.streaming=true; charset=utf-8'
    'content-encoding': 'REDACTED'
    'vary': 'REDACTED'
    'preference-applied': 'REDACTED'
    'odata-version': 'REDACTED'
    'request-id': '694f8dac-8382-11f1-a221-46d9942f8b90'
    'elapsed-time': 'REDACTED'
    'request-context': 'REDACTED'
    'date': 'Sun, 19 Jul 2026 14:59:15 GMT'
    'x-envoy-upstream-service-time': 'REDACTED'
    'strict-transport-security': 'REDACTED'
    'x-volterra-location': 'REDACTED'
    'server': 'web'
    'transfer-encoding': 'chunked'
2026-07-19 16:59:16,188 - INFO - Request URL: 'https://apis-dev.lab.technipenergies.com/sales/llmfortendering/aisearch//indexes('llm4t-documents-index-hok')/docs/search.post.search?api-version=REDACTED'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '355'
    'api-key': 'REDACTED'
    'Accept': 'ap

  [83/84] ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract_Doc Register as of 08 Aug 25.xlsx: 1 chunks
  [84/84] ROC_INPEX/Inpex AGRU CCS _FEED validation contract_Received via Ariba/ITT Part B - Information Requested by Company_Quality.docx: 1 chunks

Done. 84 files, 8572 total chunks


In [14]:
for file_name, chunks in all_files_docs.items():
    sorted_chunks = sorted(chunks, key=lambda d: extract_page_and_chunk(d['chunk_id']))
    for idx, doc in enumerate(sorted_chunks, start=1):
        doc['sequence_number'] = idx
    all_files_docs[file_name] = sorted_chunks 


In [15]:
for file_name, chunks in all_files_docs.items():
    sorted_chunks = sorted(chunks, key=lambda d: extract_page_and_chunk(d['chunk_id']))
    for idx, doc in enumerate(sorted_chunks, start=1):
        doc['sequence_number'] = idx
    all_files_docs[file_name] = sorted_chunks 

# fetch all docURLs from Neo4j
doc_url_list = []

with neo4j_driver.session() as session:
    result = session.run("MATCH (d:Document) RETURN d.docURL AS doc_url")
    for record in result:
        if record["doc_url"]:
            doc_url_list.append(record["doc_url"])

print(f"Loaded {len(doc_url_list)} docURLs from Neo4j")

def resolve_doc_url(azure_file_name: str) -> str:
    """
    azure_file_name: "ROC_INPEX/ITT_.../filename.pdf"
    docURL:          "file://c:\\Users\\hseng\\...\\ROC_INPEX\\ITT_...\\filename.pdf"
    Match: normalize slashes and check suffix.
    """
    normalized = azure_file_name.replace("/", "\\")
    for url in doc_url_list:
        if url.replace("/", "\\").endswith(normalized):
            return url
    return None

# verify
file_name_url_map = {}
for file_name in list(all_files_docs.keys()):
        url = resolve_doc_url(file_name)
        file_name_url_map[file_name] = url



Loaded 84 docURLs from Neo4j


## Step 3: Batch extraction preview

This is the inline version of the new semantic graph build path. Keep the chunk count low at first.


In [16]:
from tqdm.auto import tqdm
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from openai import APITimeoutError, APIConnectionError, RateLimitError

@retry(
    retry=retry_if_exception_type((APITimeoutError, APIConnectionError, RateLimitError)),
    wait=wait_exponential(min=2, max=60),
    stop=stop_after_attempt(5),
)
def call_llm(system_prompt, user_prompt):
    response = openai_client.chat.completions.create(
        model=chat_deployment,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
        temperature=0,
        max_tokens=16000,
        timeout=120,  # seconds
        response_format={'type': 'json_object'},
    )
    return response

In [17]:
file_name_chunks = {}

for file_name, doc_chunks in tqdm(all_files_docs.items(), desc="Processing files"):
    file_name_chunks[file_name] = len(doc_chunks)

Processing files:   0%|          | 0/84 [00:00<?, ?it/s]

In [18]:
file_names_excluded = [
    'ROC_INPEX/2.0 Tender Supplements/WS33071728 -TB-0011_Tender Supplement #5 - Updated Design Dossier (Aug25).pdf',
    'ROC_INPEX/2.0 Tender Supplements/WS33071728 -TB-0002_Tender Supplement #2 Update Design Dossier #2 & Contract Documents #1(Apr25).pdf',
    'ROC_INPEX/2.0 Tender Supplements/WS33071728 -TB-0008_Design Dossier Minor Update.pdf',
    'ROC_INPEX/2.0 Tender Supplements/WS33071728 -TB-0003_Tender Supplement #3 - Updated Design Dossier #3  Contract Documents (Exhibits)  Information response templates #2 (End April2025).pdf',
    'ROC_INPEX/2.0 Tender Supplements/WS33071728 -TB-0005_Tender Supplement #4 - Updated Design Dossier (EndMay25).pdf',
    'ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part B - Input_Exhibit B - Compensation_Rev2_20250808.xlsx',
    'ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part B - Input_Exhibit B - Compensation_Rev1_20250430.xlsx',
    'ROC_INPEX/2.0 Tender Supplements/WS33071728 -TB-0009_Company_FEED_Value_Enhancement_Optimisation_Update_.pdf',
    'ROC_INPEX/2.0 Tender Supplements/WS33071728 -TB-0001_Tender Supplement #1 - Updated Design Dossier (Jan25).pdf',
    'ROC_INPEX/2.0 Tender Supplements/WS33071728 -TB-0010_Attachment 2_VE Opportunities.pdf',
    'ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 03_Exhibit A_Scope of Work_Rev.0_20241216.pdf',
    'ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 02_Schedule 1_Generald Conditions_Rev.0_20241216.pdf',
    'ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract_Doc Register as of 08 Aug 25.xlsx',
    'ROC_INPEX/Inpex AGRU CCS _FEED validation contract_Received via Ariba/Exhibit B Tables - Commercial - Ichthys Onshore AGRU Upgrade and CCS Project FEED validation contract.xlsx',
]

In [19]:
len(file_names_excluded)

14

In [20]:
file_name_chunks_included = {}
file_name_chunks_number_included = {}
count = 0
for file_name , doc_chunks in all_files_docs.items():
    if file_name not in file_names_excluded:
        file_name_chunks_included[file_name] = doc_chunks
        file_name_chunks_number_included[file_name] = len(doc_chunks)
        count += len(doc_chunks)

In [21]:
file_name_chunks_included

{'ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 03_Exhibit A13_Contractor Documentation_Rev.1_20250808_Clean.pdf': [{'blob_url': '',
   'file_name': 'ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 03_Exhibit A13_Contractor Documentation_Rev.1_20250808_Clean.pdf',
   'chunk_type': 'text',
   'page_number': '1',
   'tender_id': 'ROC_INPEX',
   'chunk_index': 1,
   'is_active': True,
   'page_chunk': 'ICHTHYS GAS FIELD DEVELOPMENT PROJECT \n \n \nTENDER No: WS33071728 \n \nICHTHYS ONSHORE AGRU UPGRADE AND CCS PROJECT \nENGINEERING PROCUREMENT AND CONSTRUCTION \nCONTRACT \n \nEXHIBIT A13 – CONTRACTOR DOCUMENTATION',
   'mid_level_domain': ['CONSTRUCTION & COMM. START UP',
    'ENGINEERING',
    'PROCUREMENT',
    'PROJECT MANAGEMENT'],
   'high_level_domain': 'TECHNICAL',
   'chunk_id': 'ROC_INPEX_ITT_Ichthys_Onshore_AGRU_Upgrade_and_CCS_EPC_Contract_WS33071728_ITT_Part_C_-_03_Exhibit_A13_Contractor_Document

In [24]:
import json
CARDS_DIR = repo_root 
ENTITY_CARDS   = json.loads((CARDS_DIR / "entity_category_cards.json").read_text())
# RELATION_CARDS = json.loads((CARDS_DIR / "relation_cards.json").read_text())

CANONICAL_ENTITIES_CATEGORIES = list(ENTITY_CARDS.keys())   # 32
# CANONICAL_RELATION_RULES      = list(RELATION_CARDS.keys()) # 29


In [25]:
print(f"Loaded {len(CANONICAL_ENTITIES_CATEGORIES)} entity categories")

def render_entity_category_block(cards: dict) -> str:
    blocks = []
    for name, c in cards.items():
        ex = ", ".join(c.get("examples", []))
        blocks.append(
            f"- {name}\n"
            f"    definition : {c['definition']}\n"
            f"    examples   : {ex}"
        )
    return "\n".join(blocks)

# def render_relation_type_block(cards: dict) -> str:
#     blocks = []
#     for name, c in cards.items():
#         syn = ", ".join(c.get("synonyms", []))
#         blocks.append(
#             f"- {name}\n"
#             f"    synonyms   : {syn}"
#         )
#     return "\n".join(blocks)

dynamic_entity_category_block = render_entity_category_block(ENTITY_CARDS)
# dynamic_relation_type_block   = render_relation_type_block(RELATION_CARDS)


Loaded 32 entity categories


In [26]:
dynamic_entity_category_block

"- document\n    definition : A self-contained tender or contract artifact that stands alone as a deliverable — a top-level Exhibit (e.g. Exhibit A, Exhibit A7, Exhibit M), a named map, report, plan, drawing, or specification that has its own document number or title. DO NOT use for Appendices or numbered sub-sections of an Exhibit (e.g. Appendix A1.3, Appendix A1.4) — those are sections.\n    examples   : Exhibit A – Scope of Work, Exhibit A7 – Traffic Management Plan, Exhibit M – Contractor's Organisation and Key Personnel, Site Installed Plant Envelope Map, Technical Specification for CCS Compressor Package, Vendor Data Requirements List (VDRL)\n- section\n    definition : A numbered or named subdivision INSIDE a document — a chapter, clause group, appendix, or sub-exhibit whose locator is tied to its parent (e.g. Appendix A1.3 lives inside Exhibit A1). Use this for anything prefixed Appendix, Schedule, Annex, or Section when it is a sub-part of a larger document. DO NOT use for top

In [27]:
def build_semantic_extraction_prompt(
    dynamic_entity_category_block: str,
):
    return f"""
You are extracting tender-contract semantics into a knowledge graph.

Your task has 2 stages:

STAGE 1 — CHUNK CLEANING

The input chunk may contain both useful semantic content and structural noise.

You must clean the chunk before semantic extraction.

Possible cleaning outcomes:
- unchanged
  - the chunk is already substantive and should be kept as-is
- cleaned
  - the chunk contains both useful content and removable structural noise
- fully_removed
  - the chunk contains no useful semantic content after cleaning

Preserve:
- original meaning
- contractual and technical wording
- substantive legal, technical, operational, commercial, testing, compliance, evidentiary, procedural, and scope content
- section headings or labels when they are needed to interpret substantive content
- substantive content whether it appears in prose, bullets, lists, or tables

You must remove only clearly structural or non-substantive noise such as:
- document cover/title-only lines
- page number lines
- repeated running headers/footers
- separator lines
- repeated navigation labels that do not introduce substantive content

Important:
- A chunk may contain both noise and useful information.
- Do NOT exclude the whole chunk just because some lines are structural.
- Remove only the irrelevant spans and keep the substantive spans.
- Keep headings, labels, row labels, and section labels if they are needed to preserve meaning.
- Remove a heading or label only if it is acting purely as navigation or index text.


STAGE 2 — SEMANTIC EXTRACTION

Extract semantics only from cleaned_chunk, never from removed text.

Extract:
1. Entities
   - The specific actors, objects, documents, sections, requirements, values, roles, organisations, systems, locations, dates, or clauses involved.

2. Relationships
   - The semantic connections between extracted entities.
   - Relationship types are free-form.
   - Express each relationship as a complete, faithful natural-language statement.

3. Evidence spans
   - The exact text from cleaned_chunk supporting each extracted entity or relationship.


ENTITY EXTRACTION RULES

You MUST:
- reuse the canonical entity categories when they fit
- ALWAYS fill llm_category with your proposed category, even if it matches the canonical category
- if no canonical category fits, set canonical_category to "other" and preserve your proposed category type in llm_category
- only extract entities explicitly supported by cleaned_chunk
- include aliases when the source provides abbreviations, alternate names, or bracketed names
- entity description must explain the entity's role, purpose, meaning, or function in the source chunk, according to its category
- do not merely restate the entity name in the description
- description must never be empty
- if the source gives enough context, ground the description in that context
- if the source only provides a list/table entry with no explanation, write: "Listed as a [canonical_category] in the source chunk under [nearby heading/table/row context]."
Canonical entity categories with definitions and examples:
{dynamic_entity_category_block}


RELATIONSHIP EXTRACTION RULES

You MUST:
- extract relationships only between entities that appear in the entities list
- preserve directional meaning where the source implies direction
- write relationship_label as a short free-form snake_case phrase describing the relationship
- write relationship_description as a complete natural-language statement
- preserve tender-specific wording such as shall, must, requires, prior to, listed in, submitted by, approved by, incorporated into, applies to, included in, responsible for, subject to, and in accordance with when present
- prefer precise contractual, technical, procedural, commercial, schedule, approval, responsibility, inclusion, reference, or definition statements
- avoid generic relationships such as "related to" unless the source provides no clearer relationship
- include exact supporting evidence from cleaned_chunk
- assign relationship_strength as an integer from 1 to 10

Relationship strength / weight guidance:
Use the full 1-10 range. Score relationship strength relative to the other relationships in the same chunk.

- 10: Reserved for the central relationship that defines the main meaning of the chunk, such as the primary obligation, direct definition, direct listing entry, or explicit prerequisite.
- 8-9: Strong explicit relationship that is directly stated and important, but not the only central point of the chunk.
- 6-7: Clear supporting relationship, such as inclusion, reference, applicability, responsibility, schedule linkage, or document-to-document linkage.
- 4-5: Contextual relationship that is useful but secondary or indirectly stated.
- 1-3: Incidental co-occurrence or weak association. Usually do not extract these.

Normalization rules:
- Use snake_case for canonical_category, llm_category, and relationship_label.
- If there are no useful entities or relationships, return empty lists.
- llm_category is ALWAYS filled when entities are returned.
- subject and object in each relation must exactly match names from the entities list.


─────────────────────────────────────────────────────────────────────────
SELF-CHECK BEFORE RETURNING
─────────────────────────────────────────────────────────────────────────
For every entity and relation in your output, verify:
1. Every entity canonical_category is in the canonical entity list above, or is exactly "other".
   If not → set canonical_category to "other" and keep your proposed type in llm_category.

2. Every relation subject and object exactly match an entity name in the entities list.
   If not → drop the relation.

3. Every relation evidence is a substring of cleaned_chunk.
   If not → drop the relation.

4. Every relation has relationship_label, relationship_description, and relationship_strength.
   If not → drop the relation.


Return STRICT JSON only with this schema:

{{
  "cleaning_status": "unchanged" | "cleaned" | "fully_removed",
  "cleaning_reason": "short reason",
  "page_type": "substantive_clause | substantive_appendix | table_of_contents | appendix_index | cover_page | revision_history | structural_navigation | other",
  "original_chunk": "the original input chunk exactly as received",
  "cleaned_chunk": "the cleaned chunk used for extraction; empty string if fully_removed",
  "removed_spans": [
    "exact removed span 1",
    "exact removed span 2"
  ],
  "clause_summary": "short summary",
  "entities": [
    {{
      "name": "Performance Test 3",
      "canonical_category": "test",
      "llm_category": "performance_test",
      "description": "Performance Test 3 is a defined verification activity referenced in the source chunk.",
      "aliases": ["PT3"]
    }}
  ],
  "relations": [
    {{
      "subject": "Performance Test 3",
      "subject_canonical_category": "test",
      "subject_llm_category": "performance_test",
      "object": "Performance Test 1",
      "object_canonical_category": "test",
      "object_llm_category": "performance_test",
      "relationship_label": "can_only_commence_after",
      "relationship_description": "Performance Test 3 can only commence after Performance Test 1 has been successfully completed.",
      "relationship_strength": 9,
      "evidence": "Performance Test 3 can only commence after both Performance Test 1 and Performance Test 2 have been successfully completed.",
      "confidence": 0.94
    }}
  ]
}}

Rules for cleaning outcomes:
- If cleaning_status = "unchanged":
  - cleaned_chunk should be the same as original_chunk
  - removed_spans should be []
- If cleaning_status = "cleaned":
  - cleaned_chunk should keep only substantive content
  - removed_spans should list what was removed
- If cleaning_status = "fully_removed":
  - cleaned_chunk must be ""
  - entities must be []
  - relations must be []
  - clause_summary should briefly say why nothing substantive remained

Example fully removed output:
{{
  "cleaning_status": "fully_removed",
  "cleaning_reason": "The chunk contains only table of contents and navigation text.",
  "page_type": "table_of_contents",
  "original_chunk": "3 PERFORMANCE TESTS .... 3\\n4 REMEDIES .... 6",
  "cleaned_chunk": "",
  "removed_spans": [
    "3 PERFORMANCE TESTS .... 3",
    "4 REMEDIES .... 6"
  ],
  "clause_summary": "No substantive contractual content remained after cleaning.",
  "entities": [],
  "relations": []
}}

Example cleaned output:
{{
  "cleaning_status": "cleaned",
  "cleaning_reason": "Removed structural navigation lines and retained substantive testing requirements.",
  "page_type": "substantive_clause",
  "original_chunk": "3 PERFORMANCE TESTS .... 3\\nPerformance Test 3 can only commence after both Performance Test 1 and Performance Test 2 have been successfully completed.",
  "cleaned_chunk": "Performance Test 3 can only commence after both Performance Test 1 and Performance Test 2 have been successfully completed.",
  "removed_spans": [
    "3 PERFORMANCE TESTS .... 3"
  ],
  "clause_summary": "Performance Test 3 has a prerequisite relationship to Performance Test 1 and Performance Test 2.",
  "entities": [
    {{
      "name": "Performance Test 3",
      "canonical_category": "test",
      "llm_category": "performance_test",
      "description": "Performance Test 3 is a defined test activity that depends on prior successful completion of Performance Test 1 and Performance Test 2.",
      "aliases": []
    }},
    {{
      "name": "Performance Test 1",
      "canonical_category": "test",
      "llm_category": "performance_test",
      "description": "Performance Test 1 is listed in the source as a prerequisite test that must be successfully completed before Performance Test 3 can commence.",
      "aliases": []
    }},
    {{
      "name": "Performance Test 2",
      "canonical_category": "test",
      "llm_category": "performance_test",
      "description": "Performance Test 2 is listed in the source as a prerequisite test that must be successfully completed before Performance Test 3 can commence.",
      "aliases": []
    }}
  ],
  "relations": [
    {{
      "subject": "Performance Test 3",
      "subject_canonical_category": "test",
      "subject_llm_category": "performance_test",
      "object": "Performance Test 1",
      "object_canonical_category": "test",
      "object_llm_category": "performance_test",
      "relationship_label": "can_only_commence_after",
      "relationship_description": "Performance Test 3 can only commence after Performance Test 1 has been successfully completed.",
      "relationship_strength": 9,
      "evidence": "Performance Test 3 can only commence after both Performance Test 1 and Performance Test 2 have been successfully completed.",
      "confidence": 0.94
    }},
    {{
      "subject": "Performance Test 3",
      "subject_canonical_category": "test",
      "subject_llm_category": "performance_test",
      "object": "Performance Test 2",
      "object_canonical_category": "test",
      "object_llm_category": "performance_test",
      "relationship_label": "can_only_commence_after",
      "relationship_description": "Performance Test 3 can only commence after Performance Test 2 has been successfully completed.",
      "relationship_strength": 9,
      "evidence": "Performance Test 3 can only commence after both Performance Test 1 and Performance Test 2 have been successfully completed.",
      "confidence": 0.94
    }}
  ]
}}

Now clean this chunk and extract semantics from cleaned_chunk only.
"""

In [21]:

CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

STATE_FILE = CHECKPOINT_DIR / "state_closed_v4.pkl"
STATE_TMP_FILE = CHECKPOINT_DIR / "state_closed_v4.tmp.pkl"


In [23]:
import hashlib
import json
import pickle
import re
import time
from collections import Counter
from pathlib import Path
import threading
from openai import APITimeoutError, APIConnectionError
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
# -----------------------------------------------------------------------------
# CONFIG
# -----------------------------------------------------------------------------

CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

STATE_FILE = CHECKPOINT_DIR / "state_closed_v4.pkl"
STATE_TMP_FILE = CHECKPOINT_DIR / "state_closed_v4.tmp.pkl"

CHECKPOINT_EVERY_N_CHUNKS = 5
CHECKPOINT_EVERY_N_FILES = 1
PRINT_PROGRESS_EVERY_N_CHUNKS = 10


## Run in Parallel

# ---- PARALLELISM SETTINGS ----
FILE_BATCH_SIZE = 8        # number of files to process concurrently per batch
MAX_WORKERS_PER_BATCH = 8  # threads per batch (matches FILE_BATCH_SIZE)
# Adjust based on your API rate limits. If you have high limits, increase both.
# Rule of thumb: FILE_BATCH_SIZE * avg_chunks_per_file < your RPM limit
 

# -----------------------------------------------------------------------------
# STATE MANAGEMENT (thread-safe)
# -----------------------------------------------------------------------------
state_lock = threading.Lock()
 



def save_state(state: dict):
    with open(STATE_TMP_FILE, "wb") as f:
        pickle.dump(state, f)
    STATE_TMP_FILE.replace(STATE_FILE)


def init_empty_state():
    return {
        "all_entity_nodes": [],
        "all_relation_nodes": [],
        "all_chunk_payloads": [],
        "all_mentions": [],
        "all_assertions": [],
        "file_name_to_extractions": {},
        "seen_entity_keys": set(),
        "seen_relation_ids": set(),
        "seen_chunk_ids": set(),
        "seen_mention_keys": set(),
        "seen_assertion_keys": set(),
        "completed_chunk_ids": set(),
        "completed_files": set(),
        "failed_chunks": [],
        "failed_chunk_ids": set(),
        "stats": {
            "chunks_processed": 0,
            "chunks_fully_removed": 0,
            "llm_errors": 0,
            "failed_chunks": 0,
            "last_saved_at": None,
        },
    }

def load_state():
    if STATE_FILE.exists():
        with open(STATE_FILE, "rb") as f:
            state = pickle.load(f)
        print(f"Loaded checkpoint: {STATE_FILE}")
        return state
    print("No checkpoint found. Starting fresh.")
    return init_empty_state()


def maybe_save_checkpoint(state, force=False, chunk_counter_since_save=0, file_counter_since_save=0):
    should_save = (
        force
        or chunk_counter_since_save >= CHECKPOINT_EVERY_N_CHUNKS
        or file_counter_since_save >= CHECKPOINT_EVERY_N_FILES
    )
    if should_save:
        
        
        state["stats"]["last_saved_at"] = time.strftime("%Y-%m-%d %H:%M:%S")
        save_state(state)
        return True
    return False


def resolve_section_hint(chunk_text: str):
    section_hint = None
    first_line = chunk_text.splitlines()[0].strip() if chunk_text else ""
    if first_line and len(first_line) <= 140:
        if re.match(r"^(\d+(\.\d+)*)\s+.+", first_line):
            section_hint = first_line
        elif first_line.isupper() and len(first_line) <= 100:
            section_hint = first_line
    return section_hint



In [ ]:
from pathlib import Path


def fallback_entity_description(raw: dict, cleaned_chunk: str = "") -> str:
    """
    Ensure every extracted entity has a grounded, non-empty description.

    If the LLM provided a description, keep it.
    If not, use a minimal non-hallucinated fallback based on category/type.
    """
    description = (raw.get("description") or "").strip()
    if description:
        return description

    category = ((raw.get("canonical_category") or "entity").strip() or "entity").replace("_", " ")
    llm_category = ((raw.get("llm_category") or "").strip()).replace("_", " ")

    if llm_category and llm_category != category:
        return (
            f"Listed as a {category} in the source chunk, "
            f"with source-specific type '{llm_category}'."
        )

    return f"Listed as a {category} in the source chunk."


# -----------------------------------------------------------------------------
# SINGLE-CHUNK PROCESSING (called by workers, returns local results)
# -----------------------------------------------------------------------------
def process_single_chunk(
    chunk,
    file_name,
    doc_url,
    extraction_prompt,
    active_entity_categories,
    completed_chunk_ids_snapshot,
):
    """
    Process one chunk independently. Returns a result dict with all extracted
    data, without touching global state. The caller merges results under a lock.

    New relation model:
    - no canonical_relation_type
    - free-form relationship_label
    - relationship_description
    - relationship_strength
    """
    chunk_id = chunk["chunk_id"]

    # Already done (check against snapshot — final dedup happens at merge)
    if chunk_id in completed_chunk_ids_snapshot:
        return None

    chunk_text = (chunk.get("page_chunk", "") or "").strip()
    if len(chunk_text) < 40:
        return {
            "chunk_id": chunk_id,
            "skipped_short": True,
            "file_name": file_name,
        }

    section_hint = resolve_section_hint(chunk_text)

    chunk_payload = {
        "chunk_id": chunk_id,
        "doc_url": doc_url,
        "sequence_number": chunk.get("sequence_number", 0),
        "page_number": chunk.get("page_number"),
        "text": chunk_text,
        "section_hint": section_hint,
        "file_name": file_name,
    }

    # ---- LLM call (the expensive part — this is what we're parallelizing) ----
    try:
        response = call_llm(
            system_prompt=(
                "You are a Senior Legal Counsel and Procurement Specialist with deep expertise in "
                "international contract law and tender documentation. Your goal is to accurately "
                "extract knowledge graph entities and free-form relationship statements from complex "
                "tender clauses. You prioritize precision, adhere strictly to legal and technical "
                "terminology, and output your findings exclusively in valid, machine-readable JSON."
            ),
            user_prompt=(
                extraction_prompt
                + f"\n\nDOCUMENT: {file_name}\n\nCLAUSE TEXT:\n"
                + chunk_text[:9000]
            ),
        )

        if response.choices[0].finish_reason == "length":
            print(f"  WARNING: truncated {chunk_id}")

        extraction = json.loads(response.choices[0].message.content)

    except (APITimeoutError, APIConnectionError) as e:
        print(f"  TIMEOUT/CONNECTION on {chunk_id}: {e}")
        return {
            "chunk_id": chunk_id,
            "error": True,
            "error_type": type(e).__name__,
            "error_msg": str(e),
            "file_name": file_name,
            "page_number": chunk.get("page_number"),
            "text_preview": chunk_text[:500],
        }
    except Exception as e:
        print(f"  ERROR on {chunk_id}: {e}")
        return {
            "chunk_id": chunk_id,
            "error": True,
            "error_type": type(e).__name__,
            "error_msg": str(e),
            "file_name": file_name,
            "page_number": chunk.get("page_number"),
            "text_preview": chunk_text[:500],
        }

    # ---- Build local results (no global mutation) ----
    result = {
        "chunk_id": chunk_id,
        "file_name": file_name,
        "chunk_payload": chunk_payload,
        "extraction": extraction,
        "entity_nodes": [],
        "relation_nodes": [],
        "mentions": [],
        "assertions": [],
        "entity_map": {},
        "fully_removed": False,
    }

    if extraction.get("cleaning_status") == "fully_removed":
        result["fully_removed"] = True
        return result

    entity_map = {}
    entity_description_by_name = {}

    # --- Entities ---
    for raw in extraction.get("entities", []) or []:
        name = (raw.get("name") or "").strip()
        if not name:
            continue

        aliases = [
            alias.strip()
            for alias in (raw.get("aliases") or [])
            if isinstance(alias, str) and alias.strip()
        ]

        normalized_name = re.sub(r"[^a-z0-9 ]+", " ", name.lower()).strip()

        canonical_category = re.sub(
            r"[^a-z0-9 ]+", " ", (raw.get("canonical_category") or "").lower()
        ).strip().replace(" ", "_")

        llm_category = re.sub(
            r"[^a-z0-9 ]+", " ", (raw.get("llm_category") or "").lower()
        ).strip().replace(" ", "_")

        if not canonical_category or canonical_category not in active_entity_categories:
            canonical_category = "other"

        if not llm_category:
            llm_category = canonical_category

        raw_for_description = {
            **raw,
            "canonical_category": canonical_category,
            "llm_category": llm_category,
        }

        description = fallback_entity_description(
            raw_for_description,
            extraction.get("cleaned_chunk") or chunk_text,
        )

        entity_key = hashlib.sha1(
            f"entity||{normalized_name}".encode("utf-8")
        ).hexdigest()

        entity_map[name] = entity_key
        entity_description_by_name[name] = description

        result["entity_nodes"].append({
            "entity_key": entity_key,
            "name": name,
            "normalized_name": normalized_name,
            "canonical_category": canonical_category,
            "llm_category": llm_category,
            "description": description,
            "aliases": aliases,
        })

        result["mentions"].append({
            "chunk_id": chunk_id,
            "entity_key": entity_key,
        })

    # --- Relations ---
    for relation in extraction.get("relations", []) or []:
        subject_name = (relation.get("subject") or "").strip()
        object_name = (relation.get("object") or "").strip()

        relationship_label = re.sub(
            r"[^a-z0-9 ]+", " ", (relation.get("relationship_label") or "").lower()
        ).strip().replace(" ", "_")

        relationship_description = (relation.get("relationship_description") or "").strip()
        evidence_text = (relation.get("evidence") or "").strip()

        if not subject_name or not object_name:
            continue

        if not relationship_label or not relationship_description:
            continue

        subject_norm = re.sub(r"[^a-z0-9 ]+", " ", subject_name.lower()).strip()
        object_norm = re.sub(r"[^a-z0-9 ]+", " ", object_name.lower()).strip()

        subject_canonical_category = re.sub(
            r"[^a-z0-9 ]+", " ", (relation.get("subject_canonical_category") or "").lower()
        ).strip().replace(" ", "_")
        object_canonical_category = re.sub(
            r"[^a-z0-9 ]+", " ", (relation.get("object_canonical_category") or "").lower()
        ).strip().replace(" ", "_")
        subject_llm_category = re.sub(
            r"[^a-z0-9 ]+", " ", (relation.get("subject_llm_category") or "").lower()
        ).strip().replace(" ", "_")
        object_llm_category = re.sub(
            r"[^a-z0-9 ]+", " ", (relation.get("object_llm_category") or "").lower()
        ).strip().replace(" ", "_")

        subject_category = (
            subject_canonical_category
            if subject_canonical_category in active_entity_categories
            else "other"
        )
        object_category = (
            object_canonical_category
            if object_canonical_category in active_entity_categories
            else "other"
        )

        if not subject_llm_category:
            subject_llm_category = subject_category
        if not object_llm_category:
            object_llm_category = object_category

        subject_key = entity_map.get(subject_name) or hashlib.sha1(
            f"entity||{subject_norm}".encode("utf-8")
        ).hexdigest()
        object_key = entity_map.get(object_name) or hashlib.sha1(
            f"entity||{object_norm}".encode("utf-8")
        ).hexdigest()

        subject_description = entity_description_by_name.get(subject_name)
        if not subject_description:
            subject_description = fallback_entity_description({
                "name": subject_name,
                "canonical_category": subject_category,
                "llm_category": subject_llm_category,
                "description": "",
            }, extraction.get("cleaned_chunk") or chunk_text)

        object_description = entity_description_by_name.get(object_name)
        if not object_description:
            object_description = fallback_entity_description({
                "name": object_name,
                "canonical_category": object_category,
                "llm_category": object_llm_category,
                "description": "",
            }, extraction.get("cleaned_chunk") or chunk_text)

        # Subject entity node (may be new from relation)
        result["entity_nodes"].append({
            "entity_key": subject_key,
            "name": subject_name,
            "normalized_name": subject_norm,
            "canonical_category": subject_category,
            "llm_category": subject_llm_category,
            "description": subject_description,
            "aliases": [],
        })
        result["mentions"].append({
            "chunk_id": chunk_id,
            "entity_key": subject_key,
        })

        # Object entity node
        result["entity_nodes"].append({
            "entity_key": object_key,
            "name": object_name,
            "normalized_name": object_norm,
            "canonical_category": object_category,
            "llm_category": object_llm_category,
            "description": object_description,
            "aliases": [],
        })
        result["mentions"].append({
            "chunk_id": chunk_id,
            "entity_key": object_key,
        })

        try:
            confidence = float(relation.get("confidence", 0.75))
        except (TypeError, ValueError):
            confidence = 0.75
        confidence = max(0.0, min(confidence, 1.0))

        try:
            relationship_strength = int(relation.get("relationship_strength", 5))
        except (TypeError, ValueError):
            relationship_strength = 5
        relationship_strength = max(1, min(relationship_strength, 10))

        if not evidence_text:
            evidence_text = chunk_payload["text"][:1000].strip()

        relation_id = hashlib.sha1(
            (
                f"relation||{subject_key}||{relationship_label}||"
                f"{object_key}||{relationship_description.lower()}"
            ).encode("utf-8")
        ).hexdigest()

        result["relation_nodes"].append({
            "relation_id": relation_id,
            "relationship_label": relationship_label,
            "relationship_description": relationship_description,
            "relationship_strength": relationship_strength,
            "subject_key": subject_key,
            "object_key": object_key,
        })

        assertion_key = hashlib.sha1(
            f"assertion||{chunk_id}||{relation_id}".encode("utf-8")
        ).hexdigest()

        result["assertions"].append({
            "assertion_key": assertion_key,
            "chunk_id": chunk_id,
            "relation_id": relation_id,
            "evidence_text": evidence_text,
            "confidence": confidence,
            "relationship_strength": relationship_strength,
            "page_number": chunk_payload["page_number"],
            "doc_url": chunk_payload["doc_url"],
            "file_name": file_name,
        })

    result["entity_map"] = entity_map
    return result

In [ ]:

# -----------------------------------------------------------------------------
# MERGE CHUNK RESULT INTO GLOBAL STATE (under lock)
# -----------------------------------------------------------------------------
def merge_result_into_state(result, state, failed_chunks=None):
    """Thread-safe merge of one chunk's results into global state."""
    if result is None:
        return

    chunk_id = result["chunk_id"]
    file_name = result["file_name"]

    with state_lock:
        # Already merged by another thread (race)
        if chunk_id in state["completed_chunk_ids"]:
            return

        # Error result
        if result.get("error"):
            failed_record = {
                "chunk_id": chunk_id,
                "file_name": file_name,
                "page_number": result.get("page_number"),
                "error_type": result.get("error_type"),
                "error": result.get("error_msg"),
                "text_preview": result.get("text_preview", ""),
            }

            if "failed_chunks" not in state:
                state["failed_chunks"] = []
            if "failed_chunk_ids" not in state:
                state["failed_chunk_ids"] = set()

            if chunk_id not in state["failed_chunk_ids"]:
                state["failed_chunk_ids"].add(chunk_id)
                state["failed_chunks"].append(failed_record)

            if failed_chunks is not None:
                failed_chunks[file_name].append({
                    "chunk_id": chunk_id,
                    "error_type": result.get("error_type"),
                    "error": result.get("error_msg"),
                })

            state["stats"]["llm_errors"] += 1
            state["stats"]["failed_chunks"] = len(state["failed_chunks"])
            return

        # Short-text skip
        if result.get("skipped_short"):
            state["completed_chunk_ids"].add(chunk_id)
            state["stats"]["chunks_processed"] += 1
            return

        # Fully removed
        if result.get("fully_removed"):
            state["completed_chunk_ids"].add(chunk_id)
            state["stats"]["chunks_processed"] += 1
            state["stats"]["chunks_fully_removed"] += 1
            return

        # ---- Merge chunk payload ----
        if chunk_id not in state["seen_chunk_ids"]:
            state["all_chunk_payloads"].append(result["chunk_payload"])
            state["seen_chunk_ids"].add(chunk_id)

        # ---- Merge entities ----
        for ent in result["entity_nodes"]:
            if ent["entity_key"] not in state["seen_entity_keys"]:
                state["all_entity_nodes"].append(ent)
                state["seen_entity_keys"].add(ent["entity_key"])

        # ---- Merge mentions (dedup) ----
        for m in result["mentions"]:
            mention_key = f"{m['chunk_id']}||{m['entity_key']}"
            if mention_key not in state["seen_mention_keys"]:
                state["all_mentions"].append(m)
                state["seen_mention_keys"].add(mention_key)

        # ---- Merge relations ----
        for rel in result["relation_nodes"]:
            if rel["relation_id"] not in state["seen_relation_ids"]:
                state["all_relation_nodes"].append(rel)
                state["seen_relation_ids"].add(rel["relation_id"])

        # ---- Merge assertions ----
        for a in result["assertions"]:
            if a["assertion_key"] not in state["seen_assertion_keys"]:
                state["all_assertions"].append(a)
                state["seen_assertion_keys"].add(a["assertion_key"])

        state["completed_chunk_ids"].add(chunk_id)
        state["stats"]["chunks_processed"] += 1


# -----------------------------------------------------------------------------
# PROCESS ONE FILE (all its chunks, sequentially within the file)
# -----------------------------------------------------------------------------
def process_single_file(
    file_name,
    doc_chunks,
    state_snapshot,
    active_entity_categories,
    extraction_prompt,
):
    """
    Process all chunks of one file. LLM calls happen here (in the thread).
    Returns list of chunk results + file_extractions list.
    """
    doc_url = resolve_doc_url(file_name)
    if not doc_url:
        print(f"SKIP (no docURL): {file_name}")
        return None, None, file_name

    print(f"\n  [thread] Processing: {file_name.split('/')[-1]} ({len(doc_chunks)} chunks)")

    completed_snapshot = state_snapshot["completed_chunk_ids"]
    chunk_results = []
    file_extractions = []

    for chunk in doc_chunks:
        result = process_single_chunk(
            chunk=chunk,
            file_name=file_name,
            doc_url=doc_url,
            extraction_prompt=extraction_prompt,
            active_entity_categories=active_entity_categories,
            completed_chunk_ids_snapshot=completed_snapshot,
        )

        chunk_results.append(result)

        # Collect extraction for file_name_to_extractions
        if result and not result.get("error") and not result.get("skipped_short"):
            extraction = result.get("extraction")
            if extraction:
                file_extractions.append({result["chunk_id"]: extraction})

    return chunk_results, file_extractions, file_name

In [ ]:
# -----------------------------------------------------------------------------
# MAIN PARALLEL EXTRACTION LOOP  (fixed-ontology version, with per-batch logging)
# -----------------------------------------------------------------------------
import time
import json
from collections import Counter
from pathlib import Path

BATCH_LOG_DIR = Path("batch_logs/v4")
BATCH_LOG_DIR.mkdir(parents=True, exist_ok=True)


In [24]:
from collections import defaultdict

In [ ]:
def _category_distribution(items, key):
    """Count occurrences of a key across a list of dicts."""
    return dict(Counter(it.get(key) for it in items if it.get(key)).most_common())


def run_parallel_extraction(file_name_chunks_included, state, failed_chunks=None):
    """
    Divides files into batches of FILE_BATCH_SIZE and processes each batch
    concurrently using a ThreadPoolExecutor.

    Entity ontology is fixed and loaded once from entity_category_cards.json.
    Relations are free-form GraphRAG-style relationship statements:
    - relationship_label
    - relationship_description
    - relationship_strength

    After each batch, a JSON log is written to
    `output/jupyter-notebook/batch_logs/batch_<idx>.json` for monitoring.
    """
    if failed_chunks is None:
        failed_chunks = defaultdict(list)

    # ── Ensure failure tracking exists in state ──────────────────────────────
    if "failed_chunks" not in state:
        state["failed_chunks"] = []
    if "failed_chunk_ids" not in state:
        state["failed_chunk_ids"] = set()
    if "failed_chunks" not in state["stats"]:
        state["stats"]["failed_chunks"] = len(state["failed_chunks"])

    # ── Convenience aliases (refs into state dict) ───────────────────────────
    all_entity_nodes         = state["all_entity_nodes"]
    all_relation_nodes       = state["all_relation_nodes"]
    all_chunk_payloads       = state["all_chunk_payloads"]
    all_mentions             = state["all_mentions"]
    all_assertions           = state["all_assertions"]
    file_name_to_extractions = state["file_name_to_extractions"]
    completed_files          = state["completed_files"]

    # ── Build prompt + active ontology ONCE (fixed across all batches) ───────
    dynamic_entity_category_block = render_entity_category_block(ENTITY_CARDS)

    extraction_prompt = build_semantic_extraction_prompt(
        dynamic_entity_category_block=dynamic_entity_category_block
    )
    active_entity_categories = set(CANONICAL_ENTITIES_CATEGORIES)

    print(f"Fixed ontology: {len(active_entity_categories)} entity categories")
    print("Relations: free-form relationship_label + relationship_description + relationship_strength")
    print(f"Prompt size: {len(extraction_prompt):,} chars")
    print(f"Batch logs   → {BATCH_LOG_DIR}")

    # ── Build ordered list of files still to process ─────────────────────────
    pending_files = [
        (fname, chunks)
        for fname, chunks in file_name_chunks_included.items()
        if fname not in completed_files
    ]
    total_files = len(pending_files)
    print(f"\nTotal files to process: {total_files} (batch size: {FILE_BATCH_SIZE})")

    batches = [
        pending_files[i : i + FILE_BATCH_SIZE]
        for i in range(0, total_files, FILE_BATCH_SIZE)
    ]

    for batch_idx, batch in enumerate(batches):
        print(f"\n{'='*60}")
        print(f"BATCH {batch_idx + 1}/{len(batches)} — {len(batch)} files")
        print(f"{'='*60}")

        # ── Snapshot cumulative counts BEFORE the batch (to compute deltas) ──
        with state_lock:
            counts_before = {
                "entity_nodes":   len(all_entity_nodes),
                "relation_nodes": len(all_relation_nodes),
                "chunk_payloads": len(all_chunk_payloads),
                "mentions":       len(all_mentions),
                "assertions":     len(all_assertions),
                "failed_chunks":  len(state["failed_chunks"]),
            }
            state_snapshot = {
                "completed_chunk_ids": set(state["completed_chunk_ids"]),
            }

        batch_start_ts   = time.time()
        batch_file_stats = []
        batch_errors     = []

        # ---- Launch threads for this batch ----
        with ThreadPoolExecutor(max_workers=MAX_WORKERS_PER_BATCH) as executor:
            future_to_file = {}

            for fname, chunks in batch:
                future = executor.submit(
                    process_single_file,
                    file_name=fname,
                    doc_chunks=chunks,
                    state_snapshot=state_snapshot,
                    active_entity_categories=active_entity_categories,
                    extraction_prompt=extraction_prompt,
                )
                future_to_file[future] = fname

            # ---- Collect results as they complete ----
            for future in as_completed(future_to_file):
                fname = future_to_file[future]

                try:
                    chunk_results, file_extractions, file_name = future.result()
                except Exception as e:
                    print(f"  FATAL error processing file {fname}: {e}")
                    batch_errors.append({"file": fname, "error": str(e)})
                    continue

                if chunk_results is None:
                    with state_lock:
                        completed_files.add(file_name)

                    batch_file_stats.append({
                        "file": file_name,
                        "status": "no_doc_url",
                        "n_chunks": 0,
                        "n_entities": 0,
                        "n_relations": 0,
                        "n_failed_chunks": 0,
                    })
                    continue

                # Merge into global state
                for result in chunk_results:
                    merge_result_into_state(result, state, failed_chunks)

                # Per-file stats — extracted from this run only
                valid_results = [
                    r for r in chunk_results
                    if r and not r.get("error") and not r.get("skipped_short")
                ]

                n_ents = sum(len((r.get("extraction") or {}).get("entities", []) or []) for r in valid_results)
                n_rels = sum(len((r.get("extraction") or {}).get("relations", []) or []) for r in valid_results)
                n_failed = sum(1 for r in chunk_results if r and r.get("error"))

                with state_lock:
                    file_name_to_extractions[file_name] = file_extractions or []
                    completed_files.add(file_name)

                batch_file_stats.append({
                    "file": file_name,
                    "status": "ok",
                    "n_chunks": len([r for r in chunk_results if r is not None]),
                    "n_entities": n_ents,
                    "n_relations": n_rels,
                    "n_failed_chunks": n_failed,
                })

                print(
                    f"  done — {file_name.split('/')[-1]} | "
                    f"entity_nodes: {len(all_entity_nodes)} | "
                    f"relation_nodes: {len(all_relation_nodes)} | "
                    f"mentions: {len(all_mentions)} | "
                    f"assertions: {len(all_assertions)} | "
                    f"failed_chunks: {len(state['failed_chunks'])}"
                )

        batch_elapsed_s = round(time.time() - batch_start_ts, 1)

        # ── Snapshot AFTER the batch + compute deltas ────────────────────────
        with state_lock:
            counts_after = {
                "entity_nodes":   len(all_entity_nodes),
                "relation_nodes": len(all_relation_nodes),
                "chunk_payloads": len(all_chunk_payloads),
                "mentions":       len(all_mentions),
                "assertions":     len(all_assertions),
                "failed_chunks":  len(state["failed_chunks"]),
            }

            entity_dist = _category_distribution(
                all_entity_nodes,
                "canonical_category",
            )
            relation_label_dist = _category_distribution(
                all_relation_nodes,
                "relationship_label",
            )
            relation_strength_dist = _category_distribution(
                all_relation_nodes,
                "relationship_strength",
            )

        delta = {k: counts_after[k] - counts_before[k] for k in counts_before}

        # ── Save checkpoint ──────────────────────────────────────────────────
        with state_lock:
            state["stats"]["last_saved_at"] = time.strftime("%Y-%m-%d %H:%M:%S")
            state["stats"]["failed_chunks"] = len(state["failed_chunks"])
            save_state(state)

        # ── Write per-batch JSON log ─────────────────────────────────────────
        batch_log = {
            "batch_idx": batch_idx,
            "batch_label": f"{batch_idx + 1}/{len(batches)}",
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "elapsed_seconds": batch_elapsed_s,
            "n_files_in_batch": len(batch),
            "n_files_ok": sum(1 for f in batch_file_stats if f["status"] == "ok"),
            "n_files_skipped": sum(1 for f in batch_file_stats if f["status"] == "no_doc_url"),
            "n_files_errored": len(batch_errors),
            "errors": batch_errors,
            "counts_before": counts_before,
            "counts_after": counts_after,
            "delta": delta,
            "files": batch_file_stats,
            "cumulative_entity_distribution": entity_dist,
            "cumulative_relation_label_distribution": relation_label_dist,
            "cumulative_relation_strength_distribution": relation_strength_dist,
            "completed_files_total": len(completed_files),
            "completed_chunks_total": len(state["completed_chunk_ids"]),
            "failed_chunks_total": len(state["failed_chunks"]),
        }

        log_path = BATCH_LOG_DIR / f"batch_{batch_idx:04d}.json"
        log_path.write_text(json.dumps(batch_log, indent=2, ensure_ascii=False))

        print(
            f"  Checkpoint saved | batch took {batch_elapsed_s}s | "
            f"+{delta['entity_nodes']} entities, "
            f"+{delta['relation_nodes']} relations, "
            f"+{delta['failed_chunks']} failed chunks | "
            f"log: {log_path.name}"
        )

    # ── Final save + summary log ─────────────────────────────────────────────
    with state_lock:
        state["stats"]["last_saved_at"] = time.strftime("%Y-%m-%d %H:%M:%S")
        state["stats"]["failed_chunks"] = len(state["failed_chunks"])
        save_state(state)

    summary = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "total_files": total_files,
        "total_batches": len(batches),
        "total_chunk_payloads": len(all_chunk_payloads),
        "total_entity_nodes": len(all_entity_nodes),
        "total_relation_nodes": len(all_relation_nodes),
        "total_mentions": len(all_mentions),
        "total_assertions": len(all_assertions),
        "total_failed_chunks": len(state["failed_chunks"]),
        "completed_files": len(completed_files),
        "completed_chunks": len(state["completed_chunk_ids"]),
        "entity_distribution": _category_distribution(
            all_entity_nodes,
            "canonical_category",
        ),
        "relation_label_distribution": _category_distribution(
            all_relation_nodes,
            "relationship_label",
        ),
        "relation_strength_distribution": _category_distribution(
            all_relation_nodes,
            "relationship_strength",
        ),
    }

    (BATCH_LOG_DIR / "_summary.json").write_text(
        json.dumps(summary, indent=2, ensure_ascii=False)
    )

    print(f"\n{'='*60}")
    print(f"Total chunk payloads  : {summary['total_chunk_payloads']}")
    print(f"Total entity nodes    : {summary['total_entity_nodes']}")
    print(f"Total relation nodes  : {summary['total_relation_nodes']}")
    print(f"Total mentions        : {summary['total_mentions']}")
    print(f"Total assertions      : {summary['total_assertions']}")
    print(f"Failed chunks         : {summary['total_failed_chunks']}")
    print(f"Completed files       : {summary['completed_files']}")
    print(f"Completed chunks      : {summary['completed_chunks']}")
    print(f"Checkpoint saved to   : {STATE_FILE}")
    print(f"Batch logs in         : {BATCH_LOG_DIR}")
    print(f"Summary log           : {BATCH_LOG_DIR / '_summary.json'}")

In [ ]:
############# run #################
from collections import defaultdict
state = load_state()
try:
    run_parallel_extraction(file_name_chunks_included, state)
except KeyboardInterrupt:
    print("\nInterrupted by user. Saving checkpoint...")
    save_state(state)
    raise

In [ ]:
all_name_entity = []
for ele in all_entity_nodes:
    all_name_entity.append(ele['name'])

In [ ]:
with open("save_entity_name_v1.json", "w") as file:
    json.dump(all_name_entity,file, indent= 2)

In [ ]:
check = []
file_name = 'ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 11_Exhibit I_Major Subcontractors and Vendors_Rev.0_20241216.pdf'
for k,v in state['file_name_to_extractions'].items():
    if k == file_name:

        check.append(v)

In [ ]:
state['failed_chunks']

[{'chunk_id': 'ROC_INPEX_ITT_Ichthys_Onshore_AGRU_Upgrade_and_CCS_EPC_Contract_WS33071728_ITT_Part_C_-_03_Exhibit_A13_Contractor_Documentation_Rev_1_20250808_Clean_pdf_size_1459643_page_74_text_chunk_2',
  'file_name': 'ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 03_Exhibit A13_Contractor Documentation_Rev.1_20250808_Clean.pdf',
  'page_number': '74',
  'error_type': 'JSONDecodeError',
  'error': 'Unterminated string starting at: line 3779 column 5 (char 37078)',
  'text_preview': 'MAN \n2 \nContractor to supply date \n  \nA13-19-037 \nRotating \nDispatch Dossiers \nMAN \n2 \nContractor to supply date \n  \nA13-19-038 \nRotating \nManufacturing Data Records \nMDR \n2 \nContractor to supply date \nMDR including Certification Data Books \nA13-19-039 \nRotating \nMechanical Seal Drawings \nMSD \n2 \nContractor to supply date \nIncludes DGS and carbon ring seals \nA13-19-040 \nRotating \nNameplate Format Drawings \nNPF \n2 \nContractor to supply 

##### 5 hours , 74 files

In [7]:
import hashlib
import json
import pickle
import re
import time
from collections import Counter
from pathlib import Path
import threading
from openai import APITimeoutError, APIConnectionError
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
# -----------------------------------------------------------------------------
# CONFIG
# -----------------------------------------------------------------------------

CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


In [11]:
STATE_FILE = CHECKPOINT_DIR / "state_closed_v4.pkl"
def load_state():
    if STATE_FILE.exists():
        with open(STATE_FILE, "rb") as f:
            state = pickle.load(f)
        print(f"Loaded checkpoint: {STATE_FILE}")
        return state
    print("No checkpoint found. Starting fresh.")
    return init_empty_state()

In [12]:
from collections import defaultdict
# -----------------------------------------------------------------------------
# LOAD / RESUME STATE
# -----------------------------------------------------------------------------

state = load_state()

all_entity_nodes = state["all_entity_nodes"]
all_relation_nodes = state["all_relation_nodes"]
all_chunk_payloads = state["all_chunk_payloads"]
all_mentions = state["all_mentions"]
all_assertions = state["all_assertions"]
file_name_to_extractions = state["file_name_to_extractions"]

seen_entity_keys = state["seen_entity_keys"]
seen_relation_ids = state["seen_relation_ids"]
seen_chunk_ids = state["seen_chunk_ids"]
seen_mention_keys = state["seen_mention_keys"]
seen_assertion_keys = state["seen_assertion_keys"]
completed_chunk_ids = state["completed_chunk_ids"]
completed_files = state["completed_files"]

print(f"Resumed entity_nodes   : {len(all_entity_nodes)}")
print(f"Resumed relation_nodes : {len(all_relation_nodes)}")
print(f"Resumed chunk_payloads : {len(all_chunk_payloads)}")
print(f"Resumed mentions       : {len(all_mentions)}")
print(f"Resumed assertions     : {len(all_assertions)}")
print(f"Completed files        : {len(completed_files)}")
print(f"Completed chunks       : {len(completed_chunk_ids)}")


Loaded checkpoint: checkpoints/state_closed_v4.pkl
Resumed entity_nodes   : 8965
Resumed relation_nodes : 6766
Resumed chunk_payloads : 1555
Resumed mentions       : 14673
Resumed assertions     : 6833
Completed files        : 70
Completed chunks       : 1877


In [13]:
state.keys()

dict_keys(['all_entity_nodes', 'all_relation_nodes', 'all_chunk_payloads', 'all_mentions', 'all_assertions', 'file_name_to_extractions', 'seen_entity_keys', 'seen_relation_ids', 'seen_chunk_ids', 'seen_mention_keys', 'seen_assertion_keys', 'completed_chunk_ids', 'completed_files', 'failed_chunks', 'failed_chunk_ids', 'stats'])

In [14]:
state['failed_chunks']

[]

## For failed chunks

In [ ]:
# Re-extract only chunks listed in state["failed_chunks"] and merge successful results back into state.

from collections import defaultdict

# ---------------------------------------------------------------------
# 1. Prepare prompt / ontology once
# ---------------------------------------------------------------------

dynamic_entity_category_block = render_entity_category_block(ENTITY_CARDS)

extraction_prompt = build_semantic_extraction_prompt(
    dynamic_entity_category_block=dynamic_entity_category_block
)

active_entity_categories = set(CANONICAL_ENTITIES_CATEGORIES)

print(f"Prompt size: {len(extraction_prompt):,} chars")
print(f"Active entity categories: {len(active_entity_categories)}")


# ---------------------------------------------------------------------
# 2. Locate failed chunks in file_name_chunks_included
# ---------------------------------------------------------------------

failed_records = list(state.get("failed_chunks", []))
failed_chunk_ids = {
    rec.get("chunk_id")
    for rec in failed_records
    if rec.get("chunk_id")
}

print("Failed chunks to retry:", len(failed_chunk_ids))

chunk_lookup = {}

for file_name, chunks in file_name_chunks_included.items():
    for chunk in chunks:
        chunk_id = chunk.get("chunk_id")
        if chunk_id in failed_chunk_ids:
            chunk_lookup[chunk_id] = {
                "chunk": chunk,
                "file_name": file_name,
            }

missing_failed_chunk_ids = failed_chunk_ids - set(chunk_lookup)

print("Located failed chunks:", len(chunk_lookup))
print("Missing failed chunks:", len(missing_failed_chunk_ids))

if missing_failed_chunk_ids:
    print("Missing chunk IDs:")
    for cid in sorted(missing_failed_chunk_ids):
        print(" -", cid)


Prompt size: 24,512 chars
Active entity categories: 32
Failed chunks to retry: 2
Located failed chunks: 2
Missing failed chunks: 0


In [ ]:


# # ---------------------------------------------------------------------
# # 3. Remove failed chunk IDs from failed tracking before retry
# #    Successful retries stay removed; failed retries will be re-added by merge.
# # ---------------------------------------------------------------------

# with state_lock:
#     state["failed_chunks"] = [
#         rec for rec in state.get("failed_chunks", [])
#         if rec.get("chunk_id") not in failed_chunk_ids
#     ]

#     if "failed_chunk_ids" not in state:
#         state["failed_chunk_ids"] = set()

#     state["failed_chunk_ids"] = {
#         cid for cid in state["failed_chunk_ids"]
#         if cid not in failed_chunk_ids
#     }

#     # Defensive: failed chunks should not usually be completed, but remove them
#     # so process_single_chunk cannot skip them.
#     state["completed_chunk_ids"] = {
#         cid for cid in state["completed_chunk_ids"]
#         if cid not in failed_chunk_ids
#     }

#     state["stats"]["failed_chunks"] = len(state["failed_chunks"])


# # ---------------------------------------------------------------------
# # 4. Retry extraction only for located failed chunks
# # ---------------------------------------------------------------------

# retry_results = []
# retry_failed_chunks = defaultdict(list)

# for chunk_id, payload in chunk_lookup.items():
#     chunk = payload["chunk"]
#     file_name = payload["file_name"]
#     doc_url = resolve_doc_url(file_name)

#     if not doc_url:
#         result = {
#             "chunk_id": chunk_id,
#             "file_name": file_name,
#             "page_number": chunk.get("page_number"),
#             "error": True,
#             "error_type": "MissingDocURL",
#             "error_msg": f"Could not resolve doc_url for file: {file_name}",
#             "text_preview": (chunk.get("page_chunk") or "")[:500],
#         }
#     else:
#         print(f"Retrying chunk: {chunk_id}")
#         print(f"  file: {file_name}")
#         print(f"  page: {chunk.get('page_number')}")

#         result = process_single_chunk(
#             chunk=chunk,
#             file_name=file_name,
#             doc_url=doc_url,
#             extraction_prompt=extraction_prompt,
#             active_entity_categories=active_entity_categories,
#             completed_chunk_ids_snapshot=set(),
#         )

#     retry_results.append(result)


# # ---------------------------------------------------------------------
# # 5. Merge retry results into state
# # ---------------------------------------------------------------------

# for result in retry_results:
#     merge_result_into_state(result, state, retry_failed_chunks)


# # ---------------------------------------------------------------------
# # 6. Add successful retry extractions back to file_name_to_extractions
# # ---------------------------------------------------------------------

# with state_lock:
#     for result in retry_results:
#         if not result:
#             continue

#         if result.get("error") or result.get("skipped_short"):
#             continue

#         extraction = result.get("extraction")
#         if not extraction:
#             continue

#         file_name = result["file_name"]
#         chunk_id = result["chunk_id"]

#         existing = state["file_name_to_extractions"].setdefault(file_name, [])

#         # Remove old raw extraction for this chunk if it exists, then append latest.
#         cleaned_existing = []
#         for item in existing:
#             if isinstance(item, dict) and chunk_id in item:
#                 continue
#             cleaned_existing.append(item)

#         cleaned_existing.append({chunk_id: extraction})
#         state["file_name_to_extractions"][file_name] = cleaned_existing

#     state["stats"]["failed_chunks"] = len(state.get("failed_chunks", []))
#     state["stats"]["last_saved_at"] = time.strftime("%Y-%m-%d %H:%M:%S")

#     save_state(state)


# # ---------------------------------------------------------------------
# # 7. Summary
# # ---------------------------------------------------------------------

# retry_success = [
#     r for r in retry_results
#     if r and not r.get("error") and not r.get("skipped_short")
# ]

# retry_errors = [
#     r for r in retry_results
#     if r and r.get("error")
# ]

# retry_skipped = [
#     r for r in retry_results
#     if r and r.get("skipped_short")
# ]

# print("=" * 80)
# print("Retry complete")
# print("=" * 80)
# print("Retried chunks       :", len(retry_results))
# print("Successful retries   :", len(retry_success))
# print("Skipped short chunks :", len(retry_skipped))
# print("Still failed         :", len(retry_errors))
# print("State failed chunks  :", len(state.get("failed_chunks", [])))
# print("Total entity nodes   :", len(state["all_entity_nodes"]))
# print("Total relation nodes :", len(state["all_relation_nodes"]))
# print("Total mentions       :", len(state["all_mentions"]))
# print("Total assertions     :", len(state["all_assertions"]))
# print("Completed chunks     :", len(state["completed_chunk_ids"]))

# if retry_errors:
#     print("\nStill failed:")
#     for r in retry_errors:
#       print(f"- {r.get('chunk_id')} | {r.get('error_type')} | {r.get('error_msg')}")

In [ ]:
all_entity_nodes = state["all_entity_nodes"]
all_relation_nodes = state["all_relation_nodes"]
all_chunk_payloads = state["all_chunk_payloads"]
all_mentions = state["all_mentions"]
all_assertions = state["all_assertions"]
file_name_to_extractions = state["file_name_to_extractions"]

seen_entity_keys = state["seen_entity_keys"]
seen_relation_ids = state["seen_relation_ids"]
seen_chunk_ids = state["seen_chunk_ids"]
seen_mention_keys = state["seen_mention_keys"]
seen_assertion_keys = state["seen_assertion_keys"]
completed_chunk_ids = state["completed_chunk_ids"]
completed_files = state["completed_files"]

print(f"Resumed entity_nodes   : {len(all_entity_nodes)}")
print(f"Resumed relation_nodes : {len(all_relation_nodes)}")
print(f"Resumed chunk_payloads : {len(all_chunk_payloads)}")
print(f"Resumed mentions       : {len(all_mentions)}")
print(f"Resumed assertions     : {len(all_assertions)}")
print(f"Completed files        : {len(completed_files)}")
print(f"Completed chunks       : {len(completed_chunk_ids)}")


Resumed entity_nodes   : 8965
Resumed relation_nodes : 6766
Resumed chunk_payloads : 1555
Resumed mentions       : 14673
Resumed assertions     : 6833
Completed files        : 70
Completed chunks       : 1877


In [ ]:
from collections import Counter

strength_counts = Counter(
    rel.get("relationship_strength")
    for rel in all_relation_nodes
    if rel.get("relationship_strength") is not None
)

print("Relationship strength distribution:")
for strength, count in sorted(strength_counts.items(), key=lambda x: int(x[0])):
    print(f"{strength}: {count}")

Relationship strength distribution:
6: 21
7: 446
8: 2456
9: 2577
10: 1266


In [15]:
from collections import Counter, defaultdict
import pandas as pd

# Count entity-name occurrences in raw per-chunk extractions
entity_name_counter = Counter()
entity_name_files = defaultdict(set)
entity_name_chunks = defaultdict(set)
entity_name_categories = defaultdict(Counter)

for file_name, file_extractions in state["file_name_to_extractions"].items():
    for item in file_extractions:
        # item shape is usually: {chunk_id: extraction}
        for chunk_id, extraction in item.items():
            for ent in extraction.get("entities", []) or []:
                name = (ent.get("name") or "").strip()
                if not name:
                    continue

                canonical_category = ent.get("canonical_category") or "unknown"

                entity_name_counter[name] += 1
                entity_name_files[name].add(file_name)
                entity_name_chunks[name].add(chunk_id)
                entity_name_categories[name][canonical_category] += 1

rows = []
for name, count in entity_name_counter.most_common():
    rows.append({
        "entity_name": name,
        "occurrences": count,
        "file_count": len(entity_name_files[name]),
        "chunk_count": len(entity_name_chunks[name]),
        "top_category": entity_name_categories[name].most_common(1)[0][0],
        "category_distribution": dict(entity_name_categories[name]),
    })

entity_distribution_df = pd.DataFrame(rows)

print(f"Unique entity names: {len(entity_distribution_df)}")
print(f"Total entity mentions in file_name_to_extractions: {sum(entity_name_counter.values())}")

entity_distribution_df.head(100)

Unique entity names: 8617
Total entity mentions in file_name_to_extractions: 13447


,entity_name,occurrences,file_count,chunk_count,top_category,category_distribution
0,Contractor,433,31,433,role,"{'role': 424, 'organization': 9}"
1,Company,419,54,419,role,"{'organization': 85, 'role': 333, 'term': 1}"
2,Tenderer,68,29,68,role,"{'role': 66, 'term': 1, 'organization': 1}"
3,Exhibit E,67,15,67,section,"{'section': 57, 'document': 10}"
4,ILNG Facility,64,14,64,location,{'location': 64}
...,...,...,...,...,...,...
95,Exhibit H,8,6,8,section,"{'section': 7, 'document': 1}"
96,RFSU,8,2,8,milestone,"{'milestone': 7, 'term': 1}"
97,Ready for Start-up Certificates,8,3,8,approval,{'approval': 8}
98,ARIBA messaging,8,8,8,system,{'system': 8}


### Merge descriptions across entities 

In [16]:
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
import hashlib
import json
import re
import time

# ── Config ──────────────────────────────────────────────────────────────────

DESCRIPTION_MERGE_THRESHOLD = 70
MAX_DESCRIPTION_WORDS = 100
MAX_DESCRIPTION_MERGE_WORKERS = 10

ENTITY_DESCRIPTION_SUMMARY_PROMPT = """
You are a senior domain expert in EPC procurement and knowledge graph curation.
Your task is to synthesize multiple descriptions of the same entity into a single, canonical definition.

### OBJECTIVES:
1. **Canonical Synthesis:** Create a definitive, high-fidelity description that captures the entity's functional, technical, and contractual identity.
2. **Fact-Density:** Remove redundant phrasing. Focus on unique attributes, specific roles, and technical parameters present across the source descriptions.
3. **Conflict Resolution:** If descriptions conflict, prioritize technical specifications, performance requirements, or precise contractual obligations over general scope statements.
4. **Grounding:** Do not invent facts, roles, requirements, relationships, or technical parameters that are not present in the extracted descriptions.
5. **Formatting:** Write in a formal, objective, third-person style. Do not use meta-references such as "the documents state", "the descriptions mention", or "the source says".
6. **Constraint:** Max {max_length} words.

#######
Entity: {entity_name}

Extracted Descriptions:
{description_list}
#######

Return STRICT JSON only:
{{
  "description": "synthesized canonical description"
}}
""".strip()



In [ ]:

# ── Helpers ─────────────────────────────────────────────────────────────────

def normalize_entity_name(name: str) -> str:
    return re.sub(r"[^a-z0-9 ]+", " ", (name or "").lower()).strip()


def entity_key_from_name(name: str) -> str:
    normalized = normalize_entity_name(name)
    return hashlib.sha1(f"entity||{normalized}".encode("utf-8")).hexdigest()


def llm_merge_entity_description(entity_name: str, descriptions: list[str]) -> str:
    description_list = "\n".join(
        f"- {d.strip()}" for d in descriptions if d and d.strip()
    )

    prompt = ENTITY_DESCRIPTION_SUMMARY_PROMPT.format(
        entity_name=entity_name,
        description_list=description_list,
        max_length=MAX_DESCRIPTION_WORDS,
    )

    response = call_llm(
        system_prompt=(
            "You are a senior specialist in tendering, ITB analysis, EPC contracts, "
            "procurement documents, and knowledge graph curation. "
            "You output STRICT JSON only."
        ),
        user_prompt=prompt,
    )

    raw = response.choices[0].message.content or "{}"
    data = json.loads(raw)

    return (data.get("description") or "").strip()


def merge_one_entity_description(entity_key: str, entity_name: str, descriptions: list[str], occurrence_count: int) -> dict:
    try:
        merged_description = llm_merge_entity_description(
            entity_name=entity_name,
            descriptions=descriptions,
        )

        return {
            "entity_key": entity_key,
            "entity_name": entity_name,
            "occurrence_count": occurrence_count,
            "description_count": len(descriptions),
            "updated": bool(merged_description),
            "description": merged_description,
            "error": None,
        }

    except Exception as e:
        return {
            "entity_key": entity_key,
            "entity_name": entity_name,
            "occurrence_count": occurrence_count,
            "description_count": len(descriptions),
            "updated": False,
            "description": "",
            "error": str(e),
        }


# ── Collect descriptions from file_name_to_extractions ──────────────────────

entity_descriptions = defaultdict(list)
entity_occurrences = Counter()
entity_names = defaultdict(Counter)

for file_name, file_extractions in state["file_name_to_extractions"].items():
    for item in file_extractions:
        # Expected shape: {chunk_id: extraction}
        for chunk_id, extraction in item.items():
            for ent in extraction.get("entities", []) or []:
                name = (ent.get("name") or "").strip()
                if not name:
                    continue

                entity_key = entity_key_from_name(name)
                description = (ent.get("description") or "").strip()

                entity_occurrences[entity_key] += 1
                entity_names[entity_key][name] += 1

                if description:
                    entity_descriptions[entity_key].append(description)

print("entity keys with occurrences:", len(entity_occurrences))
print("entity keys with descriptions:", len(entity_descriptions))


# ── Build merge jobs: only entities with 2..70 occurrences ───────────────────

jobs = []

for entity_key, occurrence_count in entity_occurrences.items():
    if occurrence_count < 2 or occurrence_count > DESCRIPTION_MERGE_THRESHOLD:
        continue

    descriptions = entity_descriptions.get(entity_key, [])
    if not descriptions:
        continue

    entity_name = entity_names[entity_key].most_common(1)[0][0]

    jobs.append({
        "entity_key": entity_key,
        "entity_name": entity_name,
        "descriptions": descriptions,
        "occurrence_count": occurrence_count,
    })

print("description merge jobs:", len(jobs))


# ── Run LLM merges in parallel ───────────────────────────────────────────────

merge_results = []
start = time.time()

with ThreadPoolExecutor(max_workers=MAX_DESCRIPTION_MERGE_WORKERS) as executor:
    future_to_job = {
        executor.submit(
            merge_one_entity_description,
            job["entity_key"],
            job["entity_name"],
            job["descriptions"],
            job["occurrence_count"],
        ): job
        for job in jobs
    }

    for idx, future in enumerate(as_completed(future_to_job), start=1):
        result = future.result()
        merge_results.append(result)

        if result["error"]:
            print(f"[{idx}/{len(jobs)}] FAILED {result['entity_name']}: {result['error']}")
        else:
            print(f"[{idx}/{len(jobs)}] merged {result['entity_name']}")

print(f"description merge LLM phase complete in {time.time() - start:.1f}s")


# ── Update state["all_entity_nodes"] after workers finish ────────────────────

description_by_entity_key = {
    r["entity_key"]: r["description"]
    for r in merge_results
    if r.get("updated") and r.get("description")
}

updated_count = 0

for ent in state["all_entity_nodes"]:
    entity_key = ent.get("entity_key")
    if entity_key in description_by_entity_key:
        ent["description"] = description_by_entity_key[entity_key]
        updated_count += 1

failed_count = sum(1 for r in merge_results if r.get("error"))
success_count = sum(1 for r in merge_results if r.get("updated"))

state["entity_description_merge_log"] = merge_results
state["stats"]["entity_descriptions_merged_at"] = time.strftime("%Y-%m-%d %H:%M:%S")
state["stats"]["entity_descriptions_updated"] = updated_count
state["stats"]["entity_descriptions_success"] = success_count
state["stats"]["entity_descriptions_failed"] = failed_count
state["stats"]["entity_description_merge_threshold"] = DESCRIPTION_MERGE_THRESHOLD

all_entity_nodes = state["all_entity_nodes"]

print("Description merge complete")
print("jobs:", len(jobs))
print("success_count:", success_count)
print("failed_count:", failed_count)
print("updated_count in all_entity_nodes:", updated_count)
print("all_entity_nodes:", len(all_entity_nodes))

In [102]:
state['stats']

{'chunks_processed': 1877,
 'chunks_fully_removed': 313,
 'llm_errors': 2,
 'failed_chunks': 0,
 'last_saved_at': '2026-05-19 08:04:12',
 'entity_descriptions_merged_at': '2026-05-19 11:28:13',
 'entity_descriptions_updated': 1771,
 'entity_descriptions_success': 1771,
 'entity_descriptions_failed': 0,
 'entity_description_merge_threshold': 70}

In [ ]:

CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

STATE_FILE = CHECKPOINT_DIR / "state_closed_v5_merge.pkl"
with open(STATE_FILE, "wb") as f:
        pickle.dump(state, f)

#### Save Merge description to Neo4j

In [103]:
all_entity_nodes = state["all_entity_nodes"]

entity_description_rows = [
    {
        "entity_key": ent["entity_key"],
        "description": (ent.get("description") or "").strip(),
    }
    for ent in all_entity_nodes
    if ent.get("entity_key") and (ent.get("description") or "").strip()
]

In [111]:
len(entity_description_rows)

8965

In [110]:

UPDATE_ENTITY_DESCRIPTIONS_CYPHER = """
UNWIND $rows AS row
MATCH (e:Entity {entityKey: row.entity_key})
SET e.description = row.description
RETURN count(e) AS updated_entities
"""

with neo4j_driver.session() as session:
    result = session.run(
        UPDATE_ENTITY_DESCRIPTIONS_CYPHER,
        {"rows": entity_description_rows},
    ).single()

print("Updated Neo4j entity descriptions:", result["updated_entities"])

Updated Neo4j entity descriptions: 8965


In [32]:
# -----------------------------------------------------------------------------
# NEO4J WRITE — GraphRAG-style free-form relationships
# -----------------------------------------------------------------------------

WRITE_TO_NEO4J = True

if not WRITE_TO_NEO4J:
    print("WRITE_TO_NEO4J is False. Skipping graph writes.")
else:
    all_entity_nodes   = state["all_entity_nodes"]
    all_relation_nodes = state["all_relation_nodes"]
    all_mentions       = state["all_mentions"]
    all_assertions     = state["all_assertions"]

    with neo4j_driver.session() as session:
        # ---------------------------------------------------------------------
        # CONSTRAINTS / INDEXES
        # ---------------------------------------------------------------------
        session.run("""
            CREATE CONSTRAINT itb_id_unique IF NOT EXISTS
            FOR (i:ITB) REQUIRE i.ITB_ID IS UNIQUE
        """)

        session.run("""
            CREATE CONSTRAINT document_url_unique IF NOT EXISTS
            FOR (d:Document) REQUIRE d.docURL IS UNIQUE
        """)

        session.run("""
            CREATE CONSTRAINT chunk_id_unique IF NOT EXISTS
            FOR (ch:Chunk) REQUIRE ch.ChunkID IS UNIQUE
        """)

        session.run("""
            CREATE CONSTRAINT entity_key_unique IF NOT EXISTS
            FOR (e:Entity) REQUIRE e.entityKey IS UNIQUE
        """)

        session.run("""
            CREATE CONSTRAINT relation_id_unique IF NOT EXISTS
            FOR (r:Relation) REQUIRE r.relationId IS UNIQUE
        """)

        session.run("""
            CREATE INDEX entity_normalized_name IF NOT EXISTS
            FOR (e:Entity) ON (e.normalizedName)
        """)

        session.run("""
            CREATE INDEX entity_canonical_category IF NOT EXISTS
            FOR (e:Entity) ON (e.canonicalCategory)
        """)

        session.run("""
            CREATE INDEX relation_label IF NOT EXISTS
            FOR (r:Relation) ON (r.relationshipLabel)
        """)

        session.run("""
            CREATE INDEX relation_strength IF NOT EXISTS
            FOR (r:Relation) ON (r.relationshipStrength)
        """)

        # ---------------------------------------------------------------------
        # ENTITY NODES
        # ---------------------------------------------------------------------
        for entity in all_entity_nodes:
            session.run("""
                MERGE (e:Entity {entityKey: $entity_key})
                SET e.name = $name,
                    e.normalizedName = $normalized_name,
                    e.canonicalCategory = $canonical_category,
                    e.llmCategory = $llm_category,
                    e.description = $description,
                    e.aliases = $aliases
            """, {
                "entity_key": entity["entity_key"],
                "name": entity["name"],
                "normalized_name": entity["normalized_name"],
                "canonical_category": entity["canonical_category"],
                "llm_category": entity["llm_category"],
                "description": entity.get("description", ""),
                "aliases": entity.get("aliases", []),
            })

        # ---------------------------------------------------------------------
        # RELATION NODES (FREE-FORM SEMANTIC RELATIONS)
        # ---------------------------------------------------------------------
        for rel in all_relation_nodes:
            session.run("""
                MATCH (subject:Entity {entityKey: $subject_key})
                MATCH (object:Entity {entityKey: $object_key})
                MERGE (r:Relation {relationId: $relation_id})
                SET r.relationshipLabel = $relationship_label,
                    r.relationshipDescription = $relationship_description,
                    r.relationshipStrength = $relationship_strength,
                    r.weight = toFloat($relationship_strength) / 10.0,
                    r.subjectKey = $subject_key,
                    r.objectKey = $object_key
                MERGE (subject)-[:SUBJECT_OF]->(r)
                MERGE (r)-[:OBJECT_OF]->(object)
            """, {
                "relation_id": rel["relation_id"],
                "relationship_label": rel.get("relationship_label", ""),
                "relationship_description": rel.get("relationship_description", ""),
                "relationship_strength": int(rel.get("relationship_strength", 5)),
                "subject_key": rel["subject_key"],
                "object_key": rel["object_key"],
            })

        # ---------------------------------------------------------------------
        # CHUNK -> ENTITY MENTIONS
        # ---------------------------------------------------------------------
        for mention in all_mentions:
            session.run("""
                MATCH (ch:Chunk {ChunkID: $chunk_id})
                MATCH (e:Entity {entityKey: $entity_key})
                MERGE (ch)-[:MENTIONS]->(e)
            """, {
                "chunk_id": mention["chunk_id"],
                "entity_key": mention["entity_key"],
            })

        # ---------------------------------------------------------------------
        # CHUNK -> RELATION ASSERTIONS (EVIDENCE LAYER)
        # ---------------------------------------------------------------------
        for assertion in all_assertions:
            session.run("""
                MATCH (ch:Chunk {ChunkID: $chunk_id})
                MATCH (r:Relation {relationId: $relation_id})
                MERGE (ch)-[a:ASSERTS {assertionKey: $assertion_key}]->(r)
                SET a.evidenceText = $evidence_text,
                    a.confidence = $confidence,
                    a.relationshipStrength = $relationship_strength,
                    a.weight = toFloat($relationship_strength) / 10.0,
                    a.pageNumber = $page_number,
                    a.docURL = $doc_url,
                    a.fileName = $file_name
            """, {
                "assertion_key": assertion["assertion_key"],
                "chunk_id": assertion["chunk_id"],
                "relation_id": assertion["relation_id"],
                "evidence_text": assertion["evidence_text"],
                "confidence": float(assertion.get("confidence", 0.75)),
                "relationship_strength": int(assertion.get("relationship_strength", 5)),
                "page_number": assertion.get("page_number"),
                "doc_url": assertion.get("doc_url"),
                "file_name": assertion.get("file_name", ""),
            })

        # ---------------------------------------------------------------------
        # SUPPORT COUNT + AGGREGATED WEIGHT REFRESH ON RELATION NODES
        # ---------------------------------------------------------------------
        session.run("""
            MATCH (:Chunk)-[a:ASSERTS]->(r:Relation)
            WITH r,
                 count(a) AS support_count,
                 avg(coalesce(a.confidence, 0.75)) AS avg_confidence,
                 avg(coalesce(a.relationshipStrength, r.relationshipStrength, 5)) AS avg_strength
            SET r.supportCount = support_count,
                r.avgConfidence = avg_confidence,
                r.avgRelationshipStrength = avg_strength,
                r.aggregatedWeight = support_count * (avg_strength / 10.0) * avg_confidence
        """)

        # ---------------------------------------------------------------------
        # OPTIONAL: inspect counts
        # ---------------------------------------------------------------------
        summary = session.run("""
            RETURN
                count { (i:ITB) }       AS itbs,
                count { (d:Document) }  AS documents,
                count { (ch:Chunk) }    AS chunks,
                count { (e:Entity) }    AS entities,
                count { (r:Relation) }  AS relations
        """).single()

    print("Neo4j write complete")
    print(dict(summary))

2026-05-19 11:41:54,283 - INFO - Received notification from DBMS server: <GqlStatusObject gql_status='00NA0', status_description="note: successful completion - index or constraint already exists. The command 'CREATE CONSTRAINT itb_id_unique IF NOT EXISTS FOR (e:ITB) REQUIRE (e.ITB_ID) IS UNIQUE' has no effect. The index or constraint specified by 'CONSTRAINT itb_id_unique FOR (e:ITB) REQUIRE (e.ITB_ID) IS UNIQUE' already exists.", position=None, raw_classification='SCHEMA', classification=<NotificationClassification.SCHEMA: 'SCHEMA'>, raw_severity='INFORMATION', severity=<NotificationSeverity.INFORMATION: 'INFORMATION'>, diagnostic_record={'_classification': 'SCHEMA', '_severity': 'INFORMATION', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n            CREATE CONSTRAINT itb_id_unique IF NOT EXISTS\n            FOR (i:ITB) REQUIRE i.ITB_ID IS UNIQUE\n        '
2026-05-19 11:41:54,293 - INFO - Received notification from DBMS server: <GqlStatusObject gql_st

Neo4j write complete
{'itbs': 1, 'documents': 84, 'chunks': 8572, 'entities': 8965, 'relations': 6766}


relationship_keywords`: One or more high-level keywords summarizing the relationship. Multiple keywords within this field must be separated by a comma

### High-level keywords

In [15]:
with neo4j_driver.session() as s:
    ENTITY_ROWS = [row.data() for row in s.run('''
        MATCH (e:Entity)
        RETURN e.entityKey AS entity_key,
               e.name AS name,
               toLower(coalesce(e.normalizedName, '')) AS normalized_name,
               coalesce(e.canonicalCategory, 'other') AS canonical_category,
               coalesce(e.llmCategory, '') AS llm_category,
               coalesce(e.aliases, []) AS aliases
    ''')]
  
    LLM_RELATION_ROWS = [row.data() for row in s.run("""
    MATCH (r:Relation)
    MATCH (d:Document)<-[:PART_OF]-(c:Chunk)-[:ASSERTS]->(r)
    RETURN 
        d.fileName AS filename,
        r.relationshipDescription AS relationship_description,
        c.text AS chunk_text,
        r.relationId AS relation_id
""")]
    

CANON_ENTITY_CATEGORIES = sorted({row['llm_category'] for row in ENTITY_ROWS if row['canonical_category']})


In [23]:
HIGH_LEVEL_PROMPT_SYS = """You are an expert in tendering and EPC/ITB contract documents.
You extract high-level thematic keywords that describe the OVERARCHING nature of a
relationship between two entities — concepts and themes, NOT the specific entities,
numbers, or document names involved."""

HIGH_LEVEL_PROMPT_USER = """Extract relationship_keywords from the relationship below.

relationship_keywords: 3-6 high-level theme tags that summarize what KIND of
relationship this is and WHY it matters in a tender/EPC context. Focus on concepts
and themes (e.g. schedule_dependency, payment_terms, compliance_obligation,
liability_allocation, scope_inclusion, test_sequencing, approval_workflow,
document_reference), NOT the specific entities, tags, clause numbers, or values.

Rules:
- Return snake_case, lowercase tags.
- Do NOT repeat entity names from the description as keywords.
- Prefer reusable themes that would group many similar relationships together.

Relationship description:
{relationship_description}

Source chunk (context only):
{chunk}

Return STRICT JSON only:
{{"relationship_keywords": ["theme_one", "theme_two"]}}
"""


In [24]:
import re
from tqdm.auto import tqdm

def normalize_keywords(kws, label=None, cap=6):
    out = []
    for kw in kws or []:
        if not isinstance(kw, str):
            continue
        k = re.sub(r"[^a-z0-9 ]+", " ", kw.lower()).strip().replace(" ", "_")
        if k:
            out.append(k)
    out = list(dict.fromkeys(out))[:cap]     
    if not out and label:
        out = [label]                        
    return out

keywords_by_relation = {}

for rel in tqdm(LLM_RELATION_ROWS):
    rid = rel["relationID"]
    if rid in keywords_by_relation:            # resume guard
        continue

    user_prompt = HIGH_LEVEL_PROMPT_USER.format(
        relationship_description=rel["relationshipDescription"],
        chunk=(rel.get("chunk_text") or "")[:2000],
    )
    try:
        resp = call_llm(HIGH_LEVEL_PROMPT_SYS, user_prompt)
        parsed = json.loads(resp.choices[0].message.content)
        keywords_by_relation[rid] = normalize_keywords(
            parsed.get("relationship_keywords"),
            label=rel.get("relationshipLabel"),
        )
    except Exception as e:
        print(f"  ERROR {rid[:8]}: {e}")
        keywords_by_relation[rid] = normalize_keywords(None, label=rel.get("relationshipLabel"))


100%|██████████| 6833/6833 [3:04:03<00:00,  1.62s/it]


In [25]:
with neo4j_driver.session() as session:
    session.run("""
        UNWIND $rows AS row
        MATCH (r:Relation {relationId: row.relationId})
        SET r.relationshipKeywords = row.keywords
    """, {"rows": [{"relationId": k, "keywords": v}
                   for k, v in keywords_by_relation.items()]})



In [30]:
def embed_texts_large_model(texts: list[str], dimensions: int = 3072) -> list[list[float]]:
    response = openai_client.embeddings.create(
        model="text-embedding-3-large-llm4t",
        input=texts,
        dimensions=dimensions,
    )

    return [item.embedding for item in response.data]

In [26]:
# =============================================================================
# RELATION VECTOR INDEX (LightRAG "global" channel, Neo4j-native)
# =============================================================================
with neo4j_driver.session() as session:
    # ---- 0. write keywords (your existing step) ----
    session.run("""
        UNWIND $rows AS row
        MATCH (r:Relation {relationId: row.relationId})
        SET r.relationshipKeywords = row.keywords
    """, {"rows": [{"relationId": k, "keywords": v}
                   for k, v in keywords_by_relation.items()]})

    # ---- 1. pull every relation with its endpoints + description + keywords ----
    rel_rows = session.run("""
        MATCH (s:Entity)-[:SUBJECT_OF]->(r:Relation)-[:OBJECT_OF]->(o:Entity)
        RETURN r.relationId            AS relation_id,
               s.name                  AS subject,
               o.name                  AS object,
               r.relationshipDescription AS description,
               coalesce(r.relationshipKeywords, []) AS keywords
    """).data()

print(f"Relations to embed: {len(rel_rows)}")


# ---- 2. LightRAG embed-text format:  keywords \t src tgt \n description ----
def build_relation_embed_text(row):
    kw = ", ".join(row["keywords"] or [])
    return f"{kw}\t{row['subject']} {row['object']}\n{row['description']}"

embed_texts = [build_relation_embed_text(r) for r in rel_rows]



Relations to embed: 6766


In [28]:
embed_texts[0]

'value_optimization, design_dependency, scope_development, approval_workflow\tValue Enhancement (VE) process Category 1\nThe Value Enhancement (VE) process includes Category 1, which involves potential opportunities to be developed by the Tenderer based on the issued Design Dossier.'

In [31]:

# ---- 3. embed in batches with the large model ----
EMBED_BATCH = 256
relation_embeddings = []
for i in range(0, len(embed_texts), EMBED_BATCH):
    relation_embeddings.extend(embed_texts_large_model(embed_texts[i:i + EMBED_BATCH]))
    print(f"  embedded {min(i + EMBED_BATCH, len(embed_texts))}/{len(embed_texts)}")

EMBED_DIM = len(relation_embeddings[0])
print(f"Embedded {len(relation_embeddings)} relations, dim={EMBED_DIM}")


2026-06-13 14:53:15,445 - INFO - HTTP Request: POST https://apis-dev.lab.technipenergies.com/sales/llmfortendering/cognitiveservices/openai/deployments/text-embedding-3-large-llm4t/embeddings?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


  embedded 256/6766


2026-06-13 14:53:17,089 - INFO - HTTP Request: POST https://apis-dev.lab.technipenergies.com/sales/llmfortendering/cognitiveservices/openai/deployments/text-embedding-3-large-llm4t/embeddings?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


  embedded 512/6766


2026-06-13 14:53:18,443 - INFO - HTTP Request: POST https://apis-dev.lab.technipenergies.com/sales/llmfortendering/cognitiveservices/openai/deployments/text-embedding-3-large-llm4t/embeddings?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


...

  embedded 6400/6766


2026-06-13 14:53:48,659 - INFO - HTTP Request: POST https://apis-dev.lab.technipenergies.com/sales/llmfortendering/cognitiveservices/openai/deployments/text-embedding-3-large-llm4t/embeddings?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


  embedded 6656/6766


2026-06-13 14:53:50,001 - INFO - HTTP Request: POST https://apis-dev.lab.technipenergies.com/sales/llmfortendering/cognitiveservices/openai/deployments/text-embedding-3-large-llm4t/embeddings?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


  embedded 6766/6766
Embedded 6766 relations, dim=3072


In [32]:


# ---- 4. write vectors back to the Relation nodes ----
embed_rows = [
    {"relation_id": r["relation_id"], "embedding": emb}
    for r, emb in zip(rel_rows, relation_embeddings)
]
with neo4j_driver.session() as session:
    for i in range(0, len(embed_rows), 1000):
        session.run("""
            UNWIND $rows AS row
            MATCH (r:Relation {relationId: row.relation_id})
            CALL db.create.setNodeVectorProperty(r, 'embedding', row.embedding)
        """, {"rows": embed_rows[i:i + 1000]})
    print(f"Wrote embeddings to {len(embed_rows)} relations")


# ---- 5. create the vector index ----
with neo4j_driver.session() as session:
    session.run(f"""
        CREATE VECTOR INDEX relation_embedding_index IF NOT EXISTS
        FOR (r:Relation) ON (r.embedding)
        OPTIONS {{ indexConfig: {{
            `vector.dimensions`: {EMBED_DIM},
            `vector.similarity_function`: 'cosine'
        }} }}
    """)
    print("Vector index 'relation_embedding_index' ready")


Wrote embeddings to 6766 relations
Vector index 'relation_embedding_index' ready


In [21]:
file_name_chunks_included = {
    file_name.split("/", 2)[-1]: chunks
    for file_name, chunks in file_name_chunks_included.items()
}
all_file_names = list(file_name_chunks_included.keys())

In [16]:
with neo4j_driver.session() as s:
    LLM_RELATION_ROWS = [row.data() for row in s.run("""
        MATCH (d:Document)<-[:PART_OF]-(c:Chunk)-[:ASSERTS]->(r:Relation)
        WHERE d.fileName IN $all_file_names
        OPTIONAL MATCH (subj:Entity)-[:SUBJECT_OF]->(r)
        OPTIONAL MATCH (r)-[:OBJECT_OF]->(obj:Entity)
        RETURN
            d.fileName                 AS filename,
            r.relationId               AS relation_id,
            r.relationshipDescription  AS relationship_description,
            r.relationshipLabel        AS relationship_label,
            c.text                     AS chunk_text,
            subj.name                  AS subject_name,
            subj.description           AS subject_description,
            obj.name                   AS object_name,
            obj.description            AS object_description
    """, all_file_names=all_file_names)]


### Domain grouping for relations

In [ ]:
file_name_to_lower_domains = {}
file_name_to_mid_domains = {}
file_name_to_high_domains = {}
file_name_to_domains = {}
for file_name, chunks in file_name_chunks_included.items():
    for chunk in chunks:
        file_name_to_lower_domains[file_name] = chunk["low_level_domain"]
        file_name_to_mid_domains[file_name] = chunk["mid_level_domain"]
    
    

In [23]:
file_name_to_lower_domains['Contract Documents (Exhibits) & Information response templates list_20250430.xlsx']

['CONSTRUCTION AND PCC&SU',
 'DOCUMENT CONTROL',
 'PROJECT EXECUTION SYSTEMS',
 'QUALITY MANAGEMENT',
 'SUBCONTRACTING']

In [25]:
relations_by_file = defaultdict(list)
for row in LLM_RELATION_ROWS:
    relations_by_file[row["filename"]].append({
        "relation_id":             row["relation_id"],
        "relationship_description": row["relationship_description"],
        "relationship_label":       row.get("relationship_label") or "",
        "chunk_text":               row.get("chunk_text") or "",
        "subject_name":             row.get("subject_name") or "",
        "subject_description":      row.get("subject_description") or "",
        "object_name":              row.get("object_name") or "",
        "object_description":       row.get("object_description") or "",
    })


In [26]:
list(relations_by_file.keys()).__len__()

68

In [ ]:
from enum import Enum
class HighLevelDomainEnum(str, Enum):
    TECHNICAL="TECHNICAL"
    NON_TECHNICAL="NON-TECHNICAL"
    GENERAL="GENERAL"


class MidLevelDomainEnum(str, Enum):
    # Technical mid-level domains
    ENGINEERING = "ENGINEERING"
    PROCUREMENT = "PROCUREMENT"
    PROCESS_HSED = "PROCESS & HSED (SAFETY IN DESIGN)"
    PROJECT_MANAGEMENT = "PROJECT MANAGEMENT"
    CONSTRUCTION_COMM_START_UP = "CONSTRUCTION & COMM. START UP"
    
    # Non-Technical mid-level domains
    LEGAL = "LEGAL / CONTRACT"
    FINANCE = "FINANCE / TAX"
    INSURANCE = "INSURANCE"
    INTELLECTUAL_PROPERTY = "INTELLECTUAL PROPERTY / NDA"
    # General
    GENERAL = "GENERAL"

class LowLevelDomainEnum(str, Enum):
    # Engineering low-level domains
    INFORMATION_MANAGEMENT = "INFORMATION MANAGEMENT"
    CYBERSECURITY = "CYBERSECURITY"
    ROTATING = "ROTATING EQUIPMENT"
    PRESSURE_VESSELS = "PRESSURE VESSELS"
    PIPING = "PIPING INSTALLATION"
    PIPELINES = "PIPELINES"
    PACKAGE="PACKAGE"
    FIRE_EQUIPMENT="FIRED EQUIPMENT"
    ONSHORE_BUILDING_ARCHITECTURE="ONSHORE BUILDING/ ARCHITECTURE"
    CONTROL_SYSTEMS="CONTROL SYSTEMS"
    TELECOMMUNICATIONS="TELECOMMUNICATIONS"
    LABORATORY="LABORATORY"
    SUBCONTRACTING="SUBCONTRACTING"
    OFFSHORE_MOBILE_SYSTEM="OFFSHORE MOBILE SYSTEM"
    OFFSHORE_BUILDING_ARCHITECTURE="OFFSHORE BUILDING/ ARCHITECTURE"
    CIVIL_STRUCTURAL = "CIVIL / STRUCTURAL"
    ELECTRICAL = "ELECTRICAL"
    INSTRUMENTATION = "INSTRUMENTATION"
    HVAC = "HVAC"
    NAVAL_ARCHITECTURE = "NAVAL ARCHITECTURE"
    OFFSHORE_STRUCTURE = "OFFSHORE STRUCTURE"
    MATERIALS_TECHNOLOGY = "MATERIALS AND TECHNOLOGY"
    ADVANCED_SYSTEMS = "ADVANCED SYSTEMS"
    
    # Project Management low-level domains
    INFORMATION_TECHNOLOGY = "INFORMATION TECHNOLOGY"
    PROJECT_EXECUTION_SYSTEMS = "PROJECT EXECUTION SYSTEMS"
    COSTS_CONTROL = "COSTS CONTROL"
    RISK_MANAGEMENT = "RISK MANAGEMENT"
    DOCUMENT_CONTROL = "DOCUMENT CONTROL"
    HSE_MANAGEMENT = "HSE MANAGEMENT"
    SCHEDULE_MANAGEMENT = "SCHEDULE MANAGEMENT"
    QUALITY_MANAGEMENT = "QUALITY MANAGEMENT"

    
    # Construction low-level domains
    WEIGHT_CONTROL = "WEIGHT CONTROL AND MANAGEMENT"
    CONSTRUCTION_PCCSU="CONSTRUCTION AND PCC&SU"
    TRANSPORT_INSTALLATION = "TRANSPORT AND INSTALLATION"
    
    # General
    GENERAL = "GENERAL"




In [25]:
DOMAIN_DEFINITION = {
    "ENGINEERING": "Covers all engineering disciplines including design, analysis, specifications, technical drawings, and engineering systems development for industrial and infrastructure projects.",
    "PROCUREMENT": "Manages purchasing, vendor selection, material sourcing, equipment procurement, supply chain processes, RFQ/RFP management, and contract administration for project materials and services.",
    "PROCESS & HSED (SAFETY IN DESIGN)": "Combines process engineering (including PFDs, P&IDs, process descriptions, operating philosophies) with HSE Design aspects (fire protection, HAZOP, HAZID, safety-in-design reviews, and risk safeguards).",
    "PROJECT MANAGEMENT": "Covers project planning, execution, scheduling, resource management, budget control, risk management, stakeholder coordination, monitoring, reporting, and overall project governance.",
    "CONSTRUCTION & COMM. START UP": "Covers construction execution, site management, fabrication, installation, Pre-Commissioning (PCC), Commissioning, Start-Up (SU) activities, and handover procedures.",
    "LEGAL / CONTRACT": "Addresses legal matters, contract terms and conditions, regulatory compliance, agreements, obligations, rights, liability, indemnity, dispute resolution, and legal risk management.",
    "FINANCE / TAX": "Covers financial management, accounting, budgeting, cost control, invoicing, payment processing, tax compliance, fiscal reporting, revenue and expenditure tracking, and financial analysis.",
    "INSURANCE": "Manages insurance coverage, policies, premiums, claims, liability insurance, risk transfer, underwriting processes, broker relations, and compensation matters.",
    "INTELLECTUAL PROPERTY / NDA": "Protects intellectual property rights including patents, trademarks, copyrights, trade secrets, handles non-disclosure agreements, confidentiality provisions, and licensing arrangements.",
    "INFORMATION MANAGEMENT": "Manages data handling, document control, information workflows, EDMS systems, records management, databases, collaborative tools configuration, and client data-handover compliance.",
    "CYBERSECURITY": "Addresses cybersecurity, data privacy, network security, information protection, threat management, vulnerability assessments, encryption, access control, and authentication systems.",
    "ROTATING EQUIPMENT": "Covers rotating machinery including pumps, compressors, turbines, and drivers. Includes design, specification, maintenance strategies, operation procedures, and supplier qualification.",
    "PRESSURE VESSELS": "Covers design, fabrication, and inspection of pressure vessels, reactors, drums, columns, heat exchangers, storage tanks. Includes FEA, ASME code compliance, and explosion calculations.",
    "PIPING INSTALLATION": "Covers piping layouts, isometrics, material specifications, welding standards, installation procedures, stress analysis, flexibility studies, support design, and 'Issued for Construction' deliverables.",
    "PIPELINES": "Covers pipeline routing, design, flow assurance, construction, integrity management, transmission systems, subsea/onshore/offshore pipelines, and pigging operations.",
    "PACKAGE": "Covers packaged units, pre-assembled skids, modular equipment, integrated systems, vendor packages, and turnkey equipment solutions.",
    "FIRED EQUIPMENT": "Covers fired heaters, boilers, furnaces, combustion systems, burner design, thermal equipment specifications, and fuel systems.",
    "ONSHORE BUILDING/ ARCHITECTURE": "Covers onshore buildings, architectural design, facility layouts, infrastructure, civil foundations, ground-based structures, and land-based construction.",
    "CONTROL SYSTEMS": "Covers automation, SCADA, DCS, PLC systems, process control, logic programming, HMI design, and monitoring systems for industrial processes.",
    "TELECOMMUNICATIONS": "Covers telecommunications infrastructure, communication networks, radio systems, telephone systems, fiber optics, wireless communications, and data connectivity.",
    "LABORATORY": "Covers laboratory facilities, testing equipment, analysis procedures, sample management, quality testing, research capabilities, and experimental setups.",
    "SUBCONTRACTING": "Manages subcontractor scope, contracts, deliverables, coordination, yard management, construction site oversight, third-party vendor management, and outsourcing arrangements.",
    "OFFSHORE MOBILE SYSTEM": "Covers design and documentation for mobile offshore units including drilling rigs, FPSOs, floating production systems, and specialized offshore vessels.",
    "OFFSHORE BUILDING/ ARCHITECTURE": "Covers offshore living quarters, accommodation modules, offshore buildings, outfitting design, helideck layouts, and offshore facility architecture.",
    "CIVIL / STRUCTURAL": "Covers civil works, structural engineering, foundations, concrete and steel structures, frame design, bridges, buildings, roads, beams, columns, and slabs.",
    "ELECTRICAL": "Covers electrical power systems, distribution networks, cabling, switchgear, substations, transformers, power generation, lighting, load analysis, and protection systems.",
    "INSTRUMENTATION": "Covers instrumentation devices, sensors, transmitters, measurement systems, field devices, gauges, meters, calibration procedures, and control instruments.",
    "HVAC": "Covers Heating, Ventilation, and Air Conditioning system design, layouts, specifications, ductwork, chillers, cooling systems, and climate control for facilities.",
    "NAVAL ARCHITECTURE": "Covers ship and floating structure design, stability analysis, hull forms, hydrodynamic studies, buoyancy calculations, draft analysis, mooring systems, and marine operations.",
    "OFFSHORE STRUCTURE": "Covers offshore/nearshore structures, platform design, jackets, floating systems, topsides, mooring anchors, subsea structures, foundation analysis, and installation procedures.",
    "MATERIALS AND TECHNOLOGY": "Focuses on metallurgy, material science, corrosion management, coatings, material specifications, alloys, composites, material selection, testing, and emerging materials technologies.",
    "ADVANCED SYSTEMS": "Covers advanced systems engineering, digital optimization, industrial integration, AI/IoT applications, smart systems, intelligent automation, and innovative technology solutions.",
    "INFORMATION TECHNOLOGY": "Manages IT infrastructure, software and hardware systems, network administration, server management, computing resources, digital platforms, and application systems.",
    "PROJECT EXECUTION SYSTEMS": "Manages setup and administration of engineering applications including 3D modeling systems (Aveva), EDMS, collaborative tools, system administration, and software platform management.",
    "COSTS CONTROL": "Manages project costs, budget tracking, financial monitoring, expenditure control, cost estimates, variance analysis, forecasting, and cost management processes.",
    "RISK MANAGEMENT": "Covers risk identification, assessment, analysis, mitigation strategies, contingency planning, risk register management, opportunity evaluation, and risk monitoring processes.",
    "DOCUMENT CONTROL": "Manages document lifecycle, version control, revision tracking, distribution, archiving, repository management, transmittals, and document register maintenance.",
    "HSE MANAGEMENT": "Covers Health, Safety, Environment, and Security management including policies, procedures, workforce safety, emergency response, regulatory compliance, training programs, incident management, and security protocols.",
    "SCHEDULE MANAGEMENT": "Manages project schedules, timelines, milestones, planning charts, progress tracking, Gantt charts, critical path analysis, baseline management, delay analysis, and schedule coordination.",
    "QUALITY MANAGEMENT": "Covers quality plans, QA/QC procedures, audits, inspections, quality control processes, standards compliance, certification, NCR management, and quality assurance systems.",
    "WEIGHT CONTROL AND MANAGEMENT": "Covers weight estimation, monitoring, and reporting for modules and floating assets. Includes center-of-gravity calculations, mass management, weight reduction strategies, and risk-based weight change management.",
    "CONSTRUCTION AND PCC&SU": "Covers construction execution, Pre-Commissioning (PCC), Commissioning, Start-Up (SU), site management, completion activities, punch-list management, and handover procedures.",
    "TRANSPORT AND INSTALLATION": "Covers heavy-lift operations, marine transport, module installation, loadout operations, seafastening, rigging, towing of floating units, mobilization, and offshore installation procedures.",
    "GENERAL": "Covers documents that span multiple domains or don't fit into specific categories. Use when content is general overview, cross-disciplinary, or cannot be classified into other specific domains."
}
DOMAIN_KEYWORDS = {
    "ENGINEERING": [
        "engineering", "design", "technical", "specifications", "drawings",
        "calculations", "analysis", "systems", "development", "solution","electrical"
    ],

    "PROCUREMENT": [
        "procurement", "equipment", "purchasing", "supplies", "materials",
        "contracts", "vendors", "logistics", "supply-chain", "purchase-orders",
        "sourcing", "rfq", "bidding", "suppliers"
    ],

    "PROCESS & HSED (SAFETY IN DESIGN)": [
        "process", "production", "plant", "chemical", "engineering",
        "pfds", "pids", "flow", "reactors", "distillation", "philosophy",
        "hse", "design", "safety", "fire", "protection", "hazard",
        "hazop", "hazid", "risk", "safeguards", "safety-in-design"
    ],

    "PROJECT MANAGEMENT": [
        "management", "project", "planning", "execution",
        "scheduling", "risk", "budget", "resources", "deliverables",
        "coordination", "monitoring", "reporting", "governance"
    ],

    "CONSTRUCTION & COMM. START UP": [
        "construction", "pcc", "execution", "site", "procedures",
        "pre-commissioning", "commissioning", "start-up", "handover", "activities",
        "installation", "fabrication", "assembly", "testing", "completion"
    ],

    "LEGAL / CONTRACT": [
        "legal", "contract", "agreement", "terms", "conditions",
        "compliance", "regulatory", "law", "clauses", "obligations",
        "rights", "liability", "indemnity", "dispute", "resolution"
    ],

    "FINANCE / TAX": [
        "finance", "financial", "accounting", "budget", "cost",
        "invoicing", "payment", "tax", "fiscal", "revenue",
        "expenditure", "profit", "loss", "cash-flow", "roi"
    ],

    "INSURANCE": [
        "insurance", "coverage", "policy", "premium", "claim",
        "liability", "risk", "indemnity", "underwriting", "broker",
        "insurer", "benefits", "compensation"
    ],

    "INTELLECTUAL PROPERTY / NDA": [
        "intellectual", "property", "ip", "nda", "confidential",
        "non-disclosure", "proprietary", "patent", "trademark", "copyright",
        "trade-secret", "confidentiality", "licensing"
    ],

    "INFORMATION MANAGEMENT": [
        "information", "management", "systems",
        "handling", "collection", "data", "documents", "records",
        "databases", "workflows", "handover", "bid", "edms", "coordination"
    ],

    "CYBERSECURITY": [
        "cybersecurity", "cyber", "security", "data-privacy",
        "network", "protection", "firewall", "encryption", "access-control",
        "threat", "vulnerability", "intrusion", "authentication"
    ],

    "ROTATING EQUIPMENT": [
        "rotating", "machinery", "pumps", "compressors", "turbines",
        "design", "maintenance", "operation", "suppliers", "bearings",
        "seals", "drivers", "dynamic"
    ],

    "PRESSURE VESSELS": [
        "pressure", "vessels", "design", "fabrication", "inspection",
        "heat", "exchangers", "columns", "reactors", "drums", "tanks", "fea",
        "asme", "codes", "shell", "nozzle"
    ],

    "PIPING INSTALLATION": [
        "piping", "installation", "layouts", "isometrics", "material",
        "specs", "procedures", "line", "welding", "fittings", "supports",
        "routing", "stress", "analysis"
    ],

    "PIPELINES": [
        "pipelines", "pipeline", "design", "routing", "flow",
        "assurance", "construction", "integrity", "pigging", "transport",
        "transmission", "subsea", "onshore", "offshore"
    ],

    "PACKAGE": [
        "package", "packaged", "units", "skids", "modular",
        "equipment", "integrated", "pre-assembled", "vendor", "supplier"
    ],

    "FIRED EQUIPMENT": [
        "fired", "equipment", "heaters", "boilers", "furnaces",
        "combustion", "burners", "thermal", "heating", "fuel"
    ],

    "ONSHORE BUILDING/ ARCHITECTURE": [
        "onshore", "building", "architecture", "structures", "facilities",
        "design", "layout", "infrastructure", "construction", "civil",
        "foundations", "ground", "land-based"
    ],

    "CONTROL SYSTEMS": [
        "control", "systems", "automation", "scada", "dcs",
        "plc", "instrumentation", "process-control", "logic", "programming",
        "hmi", "monitoring"
    ],

    "TELECOMMUNICATIONS": [
        "telecommunications", "telecom", "communications", "network",
        "radio", "telephone", "data", "fiber", "wireless", "connectivity"
    ],

    "LABORATORY": [
        "laboratory", "lab", "testing", "analysis", "samples",
        "equipment", "instruments", "quality", "research", "experiments"
    ],

    "SUBCONTRACTING": [
        "subcontracting", "subcontractor", "outsourcing",
        "scope", "contracts", "coordination", "deliverables", "yards",
        "third-party", "vendor-management"
    ],

    "OFFSHORE MOBILE SYSTEM": [
        "offshore", "mobile", "rigs", "fpsos", "floating",
        "systems", "design", "documentation", "drilling", "units",
        "vessel", "platform"
    ],

    "OFFSHORE BUILDING/ ARCHITECTURE": [
        "offshore", "building", "architecture", "quarters", "accommodation",
        "modules", "outfitting", "design", "living", "facilities",
        "helideck", "layout"
    ],

    "CIVIL / STRUCTURAL": [
        "civil", "structural", "works", "architecture", "foundations",
        "concrete", "steel", "structures", "bridges", "buildings", "roads",
        "frame", "beam", "column", "slab"
    ],

    "ELECTRICAL": [
        "electrical", "power", "distribution", "cabling",
        "switchgear", "substations", "transformers", "powergrid", "generation", "lighting",
        "load", "voltage", "current", "protection"
    ],

    "INSTRUMENTATION": [
        "instrumentation", "instruments", "sensors", "transmitters",
        "measurement", "control", "devices", "gauges", "meters",
        "field-devices", "calibration"
    ],

    "HVAC": [
        "hvac", "heating", "ventilation", "air", "conditioning",
        "systems", "design", "specifications", "ducts", "chillers",
        "cooling", "climate-control"
    ],

    "NAVAL ARCHITECTURE": [
        "naval", "architecture", "ship", "floating", "structure",
        "design", "stability", "analysis", "vessels", "marine", "hydrodynamics",
        "hull", "buoyancy", "draft"
    ],

    "OFFSHORE STRUCTURE": [
        "offshore", "structure", "platform", "subsea", "design",
        "analysis", "jackets", "floating", "modules", "topsides", "installation",
        "foundation", "mooring"
    ],

    "MATERIALS AND TECHNOLOGY": [
        "materials", "technology", "metallurgy", "corrosion",
        "specifications", "coatings", "alloys", "composites", "innovation",
        "properties", "selection", "testing"
    ],

    "ADVANCED SYSTEMS": [
        "advanced", "systems", "digital", "optimization",
        "industrial", "integration", "technology", "innovation",
        "ai", "iot", "smart", "intelligent"
    ],

    "INFORMATION TECHNOLOGY": [
        "information", "technology", "it", "software", "hardware",
        "systems", "network", "infrastructure", "applications", "servers",
        "computing", "digital"
    ],

    "PROJECT EXECUTION SYSTEMS": [
        "execution", "systems", "administration", "applications", "modeling",
        "engineering", "process", "information", "proposal", "software",
        "tools", "platforms", "aveva", "3d"
    ],

    "COSTS CONTROL": [
        "costs", "control", "budget", "financial", "tracking",
        "monitoring", "expenditure", "estimates", "variance", "forecasting",
        "cost-management"
    ],

    "RISK MANAGEMENT": [
        "risk", "management", "assessment", "mitigation", "identification",
        "analysis", "contingency", "threats", "opportunities", "evaluation",
        "register", "monitoring"
    ],

    "DOCUMENT CONTROL": [
        "document", "control", "management", "records", "filing",
        "version", "revision", "distribution", "archive", "repository",
        "transmittal", "register"
    ],

    "HSE MANAGEMENT": [
        "hse", "management", "safety", "health", "environment",
        "policies", "procedures", "workforce", "emergency", "regulations",
        "training", "compliance", "incident", "security"
    ],

    "SCHEDULE MANAGEMENT": [
        "schedule", "management", "planning", "timelines", "milestones",
        "charts", "progress", "tracking", "gantt", "critical-path",
        "baseline", "delay", "coordination"
    ],

    "QUALITY MANAGEMENT": [
        "quality", "management", "plans", "audits", "inspections",
        "control", "policies", "assurance", "standards", "certification", "compliance",
        "qaqc", "procedures", "ncr"
    ],

    "WEIGHT CONTROL AND MANAGEMENT": [
        "weight", "control", "management", "estimation", "reporting",
        "monitoring", "structures", "modules", "lifting", "balance",
        "center-of-gravity", "mass"
    ],

    "CONSTRUCTION AND PCC&SU": [
        "construction", "pcc", "pcsu", "execution", "site", "procedures",
        "pre-commissioning", "commissioning", "start-up", "handover", "activities",
        "completion", "punch-list"
    ],

    "TRANSPORT AND INSTALLATION": [
        "installation", "transport", "marine", "lifting", "modules",
        "heavy-lift", "operations", "cargo", "mobilization", "towing",
        "loadout", "seafastening", "rigging"
    ],

    "GENERAL": [
        "general", "miscellaneous", "overview", "summary", "introduction",
        "unknown", "none", "unclassified", "other", "various"
    ]
}

# MID_LEVEL_DOMAIN_DEFINITIONS = {
#     "ENGINEERING": (
#         "Covers technical engineering scope, design basis, calculations, specifications, drawings, "
#         "models, discipline engineering deliverables, technical standards, design reviews, interfaces, "
#         "and engineering requirements for facilities, equipment, systems, and materials in an EPC oil & gas project."
#     ),

#     "PROCUREMENT": (
#         "Covers purchasing, sourcing, vendor/subcontractor management, material requisitions, bid evaluation, "
#         "supplier qualification, purchase orders, expediting, logistics, inspection coordination, and delivery "
#         "of equipment, bulk materials, packages, and services required for the EPC works."
#     ),

#     "PROCESS & HSED (SAFETY IN DESIGN)": (
#         "Covers process engineering, process safety, health, safety, environment, and design safety requirements. "
#         "Includes process design, HAZID, HAZOP, SIL, safety studies, risk assessments, environmental requirements, "
#         "loss prevention, hazardous area classification, and safety-in-design obligations."
#     ),

#     "PROJECT MANAGEMENT": (
#         "Covers overall project execution management, planning, scheduling, progress reporting, project controls, "
#         "coordination, document control, quality management, risk management, interface management, change management, "
#         "communication, meetings, and general execution plans required to manage the EPC contract."
#     ),

#     "CONSTRUCTION & COMM. START UP": (
#         "Covers site construction, fabrication, installation, pre-commissioning, commissioning, start-up, handover, "
#         "construction methods, temporary facilities, site supervision, completions, testing, punch lists, and readiness "
#         "activities required to bring the facility into operation."
#     ),

#     "LEGAL / CONTRACT": (
#         "Covers contractual terms, obligations, rights, liabilities, claims, notices, amendments, contract interpretation, "
#         "governing law, dispute resolution, compliance with contract documents, warranties, indemnities, termination, "
#         "assignment, confidentiality obligations, and formal legal responsibilities between the parties."
#     ),

#     "FINANCE / TAX": (
#         "Covers commercial and financial matters including pricing, payment terms, invoicing, taxes, duties, bonds, "
#         "securities, guarantees, currency, cost breakdowns, reimbursement, audit rights, financial reporting, "
#         "liquidated damages, and other monetary obligations under the EPC tender or contract."
#     ),

#     "INSURANCE": (
#         "Covers insurance requirements, policies, coverage limits, certificates of insurance, insured parties, "
#         "deductibles, exclusions, claims handling, contractor insurance obligations, project insurance, and compliance "
#         "with required insurance provisions under the EPC contract."
#     ),

#     "INTELLECTUAL PROPERTY / NDA": (
#         "Covers ownership, use, protection, licensing, confidentiality, and disclosure of intellectual property, "
#         "technical data, proprietary information, trade secrets, software, designs, know-how, and non-disclosure "
#         "obligations during tendering and EPC execution."
#     ),

#     "GENERAL": (
#         "Covers documents that span multiple domains or don't fit into specific categories. Use when content is general overview, cross-disciplinary, or cannot be classified into other specific domains."
#     ),
# }

In [24]:
domain_hierarchy = {
    "TECHNICAL": {
        "ENGINEERING": [
            "INFORMATION MANAGEMENT",
            "CYBERSECURITY",
            "ROTATING EQUIPMENT",
            "PRESSURE VESSELS",
            "PIPING INSTALLATION",
            "PIPELINES",
            "PACKAGE",
            "FIRED EQUIPMENT",
            "ONSHORE BUILDING/ ARCHITECTURE",
            "CONTROL SYSTEMS",
            "TELECOMMUNICATIONS",
            "LABORATORY",
            "SUBCONTRACTING",
            "OFFSHORE MOBILE SYSTEM",
            "OFFSHORE BUILDING/ ARCHITECTURE",
            "CIVIL / STRUCTURAL",
            "ELECTRICAL",
            "INSTRUMENTATION",
            "HVAC",
            "NAVAL ARCHITECTURE",
            "OFFSHORE STRUCTURE",
            "MATERIALS AND TECHNOLOGY",
            "ADVANCED SYSTEMS",
        ],
        "PROCUREMENT": [],
        "PROCESS & HSED (SAFETY IN DESIGN)": [],
        "PROJECT MANAGEMENT": [
            "INFORMATION TECHNOLOGY",
            "PROJECT EXECUTION SYSTEMS",
            "COSTS CONTROL",
            "RISK MANAGEMENT",
            "DOCUMENT CONTROL",
            "HSE MANAGEMENT",
            "SCHEDULE MANAGEMENT",
            "QUALITY MANAGEMENT",
        ],
        "CONSTRUCTION & COMM. START UP": [
            "WEIGHT CONTROL AND MANAGEMENT",
            "CONSTRUCTION AND PCC&SU",
            "TRANSPORT AND INSTALLATION",
        ],
    },
    "NON-TECHNICAL": {
        "LEGAL / CONTRACT": [],
        "FINANCE / TAX": [],
        "INSURANCE": [],
        "INTELLECTUAL PROPERTY / NDA": [],
    },
    "GENERAL": {
        "GENERAL": [],
    },
}

In [28]:
CLASSIFY_LOWER_PROMPT_SYS = """
You are an expert in tendering and EPC/ITB oil & gas contract documents.
Classify ONE graph relationship into a lower domain or GENERAL, based on the
SEMANTIC MEANING of the relationship — using its label, the subject/object
entities, and the source clause as context.
"""

CLASSIFY_LOWER_PROMPT_USER = """
Available lower domains:
{lower_domains}

Domain definitions (keywords are signals only — do NOT classify on a keyword alone):
{lower_domain_definitions}

Rules:
1. Classify by the relationship's meaning, not by surface keywords.
2. Use the label and subject/object entities to disambiguate.
3. If no domain genuinely fits, return "GENERAL".

--- Relationship to classify ---
Subject: {subject_name} — {subject_description}
Object:  {object_name} — {object_description}
Relationship description: {relationship_description}
Source clause:
{chunk_text}

Return STRICT JSON only:
{{"reasoning": "<1-2 sentences>", "lower_domain": "<one lower domain or GENERAL>"}}
"""

CLASSIFY_MID_PROMPT_SYS = """
You are an expert in tendering and EPC/ITB oil & gas contract documents.
Classify ONE graph relationship into a mid-level domain or GENERAL, based on the
SEMANTIC MEANING of the relationship — using its label, the subject/object
entities, and the source clause as context.
"""

CLASSIFY_MID_PROMPT_USER = """
Available mid-level domains:
{mid_domains}

Domain definitions (keywords are signals only — do NOT classify on a keyword alone):
{mid_domain_definitions}

Rules:
1. Classify by the relationship's meaning, not by surface keywords.
2. Use the label and subject/object entities to disambiguate.
3. If no domain genuinely fits, return "GENERAL".

--- Relationship to classify ---
Subject: {subject_name} — {subject_description}
Object:  {object_name} — {object_description}
Relationship description: {relationship_description}
Source clause:
{chunk_text}

Return STRICT JSON only:
{{"reasoning": "<1-2 sentences>", "mid_domain": "<one mid domain or GENERAL>"}}
"""


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading, json

MAX_CLASSIFY_WORKERS = 8
CHUNK_CHARS = 1200          # truncate source clause

# ── optional per-domain keywords (signals only) ──
DOMAIN_KEYWORDS = {}
kw_path = ai_services_dir / "src/document_classification/domain_keywords.json"
if kw_path.exists():
    DOMAIN_KEYWORDS = json.loads(kw_path.read_text())

def render_definitions(domains, definitions):
    lines = []
    for d in domains:
        block = f"- {d}: {definitions.get(d, '')}"
        kws = DOMAIN_KEYWORDS.get(d)
        if kws:
            block += f"\n    keywords: {', '.join(kws[:8])}"
        lines.append(block)
    return "\n".join(lines)

# ── 1. Per-file config (no LLM calls) ──
file_configs = {}
for file_name in relations_by_file:
    lower_domains = file_name_to_lower_domains[file_name]
    if lower_domains == []:
        domains = list(file_name_to_mid_domains[file_name]) + ["GENERAL"]
        file_configs[file_name] = {
            "key": "mid_domain",
            "sys": CLASSIFY_MID_PROMPT_SYS,
            "template": CLASSIFY_MID_PROMPT_USER,
            "domains": domains,
            "defs_block": render_definitions(domains, MID_LEVEL_DOMAIN_DEFINITIONS),
            "domains_field": "mid_domains",
            "defs_field": "mid_domain_definitions",
        }
    else:
        domains = list(lower_domains) + ["GENERAL"]
        file_configs[file_name] = {
            "key": "lower_domain",
            "sys": CLASSIFY_LOWER_PROMPT_SYS,
            "template": CLASSIFY_LOWER_PROMPT_USER,
            "domains": domains,
            "defs_block": render_definitions(domains, LOWER_DEFINITION),
            "domains_field": "lower_domains",
            "defs_field": "lower_domain_definitions",
        }

# ── 2. Worker ──
def classify_relation(file_name, rel):
    cfg = file_configs[file_name]
    user_prompt = cfg["template"].format(
        **{cfg["domains_field"]: cfg["domains"],
           cfg["defs_field"]: cfg["defs_block"]},
        relationship_label=rel["relationship_label"],
        subject_name=rel["subject_name"],
        subject_description=rel["subject_description"],
        object_name=rel["object_name"],
        object_description=rel["object_description"],
        relationship_description=rel["relationship_description"],
        chunk_text=rel["chunk_text"][:CHUNK_CHARS],
    )
    resp = call_llm(cfg["sys"], user_prompt)
    parsed = json.loads(resp.choices[0].message.content)
    return (
        file_name,
        rel["relationship_description"],
        cfg["key"],
        parsed[cfg["key"]],
        parsed.get("reasoning", ""),
    )

# ── 3. Run all (file, relation) pairs in parallel ──
final_result = {fn: {} for fn in relations_by_file}
tasks = [(fn, rel) for fn, rels in relations_by_file.items() for rel in rels]
total = len(tasks)
lock = threading.Lock()
done = 0

with ThreadPoolExecutor(max_workers=MAX_CLASSIFY_WORKERS) as executor:
    future_to_task = {executor.submit(classify_relation, fn, rel): (fn, rel)
                      for fn, rel in tasks}
    for future in as_completed(future_to_task):
        fn, rel = future_to_task[future]
        try:
            f, desc, key, domain, reasoning = future.result()
            with lock:
                final_result[f][desc] = {key: domain, "reasoning": reasoning}
        except Exception as e:
            print(f"  ERROR {rel['relationship_description'][:40]}: {e}")
        with lock:
            done += 1
        print(f"{done}/{total}")


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading, json

MAX_CLASSIFY_WORKERS = 8
CHUNK_CHARS = 1200          # truncate source clause

# ── optional per-domain keywords (signals only) ──
DOMAIN_KEYWORDS = {}
kw_path = ai_services_dir / "src/document_classification/domain_keywords.json"
if kw_path.exists():
    DOMAIN_KEYWORDS = json.loads(kw_path.read_text())

def render_definitions(domains, definitions):
    lines = []
    for d in domains:
        block = f"- {d}: {definitions.get(d, '')}"
        kws = DOMAIN_KEYWORDS.get(d)
        if kws:
            block += f"\n    keywords: {', '.join(kws[:8])}"
        lines.append(block)
    return "\n".join(lines)

# ── 1. Per-file config (no LLM calls) ──
file_configs = {}
for file_name in relations_by_file:
    lower_domains = file_name_to_lower_domains[file_name]
    if lower_domains == []:
        domains = list(file_name_to_mid_domains[file_name]) + ["GENERAL"]
        file_configs[file_name] = {
            "key": "mid_domain",
            "sys": CLASSIFY_MID_PROMPT_SYS,
            "template": CLASSIFY_MID_PROMPT_USER,
            "domains": domains,
            "defs_block": render_definitions(domains, MID_LEVEL_DOMAIN_DEFINITIONS),
            "domains_field": "mid_domains",
            "defs_field": "mid_domain_definitions",
        }
    else:
        domains = list(lower_domains) + ["GENERAL"]
        file_configs[file_name] = {
            "key": "lower_domain",
            "sys": CLASSIFY_LOWER_PROMPT_SYS,
            "template": CLASSIFY_LOWER_PROMPT_USER,
            "domains": domains,
            "defs_block": render_definitions(domains, LOWER_DEFINITION),
            "domains_field": "lower_domains",
            "defs_field": "lower_domain_definitions",
        }

# ── 2. Worker ──
def classify_relation(file_name, rel):
    cfg = file_configs[file_name]
    user_prompt = cfg["template"].format(
        **{cfg["domains_field"]: cfg["domains"],
           cfg["defs_field"]: cfg["defs_block"]},
        subject_name=rel["subject_name"],
        subject_description=rel["subject_description"],
        object_name=rel["object_name"],
        object_description=rel["object_description"],
        relationship_description=rel["relationship_description"],
        chunk_text=rel["chunk_text"][:CHUNK_CHARS],
    )
    resp = call_llm(cfg["sys"], user_prompt)
    parsed = json.loads(resp.choices[0].message.content)
    return file_name, rel["relationship_description"], parsed[cfg["key"]]

# ── 3. Run all (file, relation) pairs in parallel ──
final_result = {fn: {} for fn in relations_by_file}
tasks = [(fn, rel) for fn, rels in relations_by_file.items() for rel in rels]
total = len(tasks)
lock = threading.Lock()
done = 0

with ThreadPoolExecutor(max_workers=MAX_CLASSIFY_WORKERS) as executor:
    future_to_task = {executor.submit(classify_relation, fn, rel): (fn, rel)
                      for fn, rel in tasks}
    for future in as_completed(future_to_task):
        fn, rel = future_to_task[future]
        try:
            f, desc, domain = future.result()
            with lock:
                final_result[f][desc] = {cfg_key: domain
                                         for cfg_key in [file_configs[f]["key"]]}
        except Exception as e:
            print(f"  ERROR {rel['relationship_description'][:40]}: {e}")
        with lock:
            done += 1
        print(f"{done}/{total}")


In [ ]:
# CLASSIFY_LOWER_PROMPT_SYS = """
# You are an expert in tendering and EPC/ITB oil & gas contract documents.
# Your task is to classify ONE graph relationship instance into a lower domain or a GENERAL domain.
# You must classify based on the semantic meaning of the relationship description.

# """

# CLASSIFY_LOWER_PROMPT_USER = """

# Available lower domains:
# {lower_domains}

# Domain definitions:
# {lower_domain_definitions}


# Classification rules:
# 1. Classify the RELATIONSHIP DESCRIPTION
# 2. Do not choose a domain only because a keyword appears in the text.
# 3. If none of the provided domains fit, return "GENERAL". 

# Relationship description:
# {relationship_description}



# Return STRICT JSON only:
# {{"lower_domain": "<one lower domain or GENERAL>"}}
# """

# CLASSIFY_MID_PROMPT_SYS = """
# You are an expert in tendering and EPC/ITB oil & gas contract documents.

# Your task is to classify ONE graph relationship instance into a middle domain or a GENERAL domain.
# You must classify based on the semantic meaning of the relationship description.

# """

# CLASSIFY_MID_PROMPT_USER = """

# Available lower domains:
# {mid_domains}

# Domain definitions:
# {mid_domain_definitions}

# Classification rules:
# 1. Classify the RELATIONSHIP DESCRIPTION
# 2. Do not choose a domain only because a keyword appears in the text.
# 3. If none of the provided domains fit, return "GENERAL". 

# Relationship description:
# {relationship_description}

# Return STRICT JSON only:
# {{"mid_domain": "<one lower domain or GENERAL>"}}
# """



In [75]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

MAX_CLASSIFY_WORKERS = 8   # raise toward your API RPM limit

# ── 1. Precompute per-file domain config once (no LLM calls here) ──
file_configs = {}
for file_name in relations_by_file:
    lower_domains = file_name_to_lower_domains[file_name]

    if lower_domains == []:
        mid_domains = list(file_name_to_mid_domains[file_name]) + ["GENERAL"]
        defs = {d: MID_LEVEL_DOMAIN_DEFINITIONS[d] for d in mid_domains}
        file_configs[file_name] = {
            "level": "mid",
            "sys": CLASSIFY_MID_PROMPT_SYS,
            "template": CLASSIFY_MID_PROMPT_USER,
            "domains": mid_domains,
            "defs": defs,
        }
    else:
        lower_domains = list(lower_domains) + ["GENERAL"]
        defs = {d: LOWER_DEFINITION[d] for d in lower_domains}
        file_configs[file_name] = {
            "level": "lower",
            "sys": CLASSIFY_LOWER_PROMPT_SYS,
            "template": CLASSIFY_LOWER_PROMPT_USER,
            "domains": lower_domains,
            "defs": defs,
        }


In [ ]:

# ── 2. Worker: classify a single relation ──
def classify_relation(file_name, rel):
    cfg = file_configs[file_name]
    desc = rel["relationship_description"]

    if cfg["level"] == "mid":
        user_prompt = cfg["template"].format(
            relationship_description=desc,
            mid_domains=cfg["domains"],
            mid_domain_definitions=cfg["defs"],
        )
    else:
        user_prompt = cfg["template"].format(
            relationship_description=desc,
            lower_domains=cfg["domains"],
            lower_domain_definitions=cfg["defs"],
        )

    resp = call_llm(cfg["sys"], user_prompt)
    return file_name, desc, json.loads(resp.choices[0].message.content)

# ── 3. Flatten all (file, relation) pairs and run in parallel ──
final_result = {fn: {} for fn in relations_by_file}
tasks = [
    (file_name, rel)
    for file_name, relations in relations_by_file.items()
    for rel in relations
]
total = len(tasks)
result_lock = threading.Lock()
done = 0

with ThreadPoolExecutor(max_workers=MAX_CLASSIFY_WORKERS) as executor:
    future_to_task = {
        executor.submit(classify_relation, file_name, rel): (file_name, rel)
        for file_name, rel in tasks
    }

    for future in as_completed(future_to_task):
        file_name, rel = future_to_task[future]
        try:
            fn, desc, parsed = future.result()
            with result_lock:
                final_result[fn][desc] = parsed
        except Exception as e:
            print(f"  ERROR {rel['relationship_description'][:40]}: {e}")

        with result_lock:
            done += 1
        print(f"{done}/{total}")


In [ ]:
# final_result = {}
# for file_name, relations in relations_by_file.items():
#     relation_count = len(relations)
#     index = 0
#     lower_domains = file_name_to_lower_domains[file_name]

#     result = {}
#     ########################

#     if lower_domains == []:
#         mid_domain_definitions = {}
#         mid_domains = list(file_name_to_mid_domains[file_name]) + ["GENERAL"]

#         for domain in mid_domains:
#             mid_domain_definitions[domain] = MID_LEVEL_DOMAIN_DEFINITIONS[domain]

#         for rel in relations:
#             user_prompt = CLASSIFY_MID_PROMPT_USER.format(
#                 relationship_description=rel["relationship_description"],
#                 mid_domains=mid_domains,
#                 mid_domain_definitions=mid_domain_definitions,
#             )
#             try:
#                 resp = call_llm(CLASSIFY_MID_PROMPT_SYS, user_prompt)
#                 parsed = json.loads(resp.choices[0].message.content)
#                 result[rel["relationship_description"]] = parsed
#             except Exception as e:
#                 print(f"  ERROR {rel['relationship_description'][:40]}: {e}")
#             index += 1
#             print(f"{index}/{relation_count}")
#     else:
#         lower_domain_definitions = {}
#         lower_domains = list(lower_domains) + ["GENERAL"]
#         for domain in lower_domains:
#             lower_domain_definitions[domain] = LOWER_DEFINITION[domain]

#         for rel in relations:
#             user_prompt = CLASSIFY_LOWER_PROMPT_USER.format(
#                 relationship_description=rel["relationship_description"],
#                 lower_domains=lower_domains,
#                 lower_domain_definitions=lower_domain_definitions,
#             )
#             try:
#                 resp = call_llm(CLASSIFY_LOWER_PROMPT_SYS, user_prompt)
#                 parsed = json.loads(resp.choices[0].message.content)
#                 result[rel["relationship_description"]] = parsed
#             except Exception as e:
#                 print(f"  ERROR {rel['relationship_description'][:40]}: {e}")
#             index += 1
#             print(f"{index}/{relation_count}")

#     final_result[file_name] = result


### Insert to Neo4J

In [33]:
DOMAIN_PROP = "domainName"

basename_to_url = {}
collisions = {}
for path, url in file_name_url_map.items():
    base = path.split("/")[-1]
    if base in basename_to_url:
        collisions.setdefault(base, []).append(path)
    basename_to_url[base] = url

if collisions:
    print(f"WARNING: {len(collisions)} duplicate basenames across folders "
          f"(last one wins): {list(collisions)[:5]}")


In [35]:
# ── 1. Flatten final_result into per-level rows ──
lower_rows, mid_rows, missing_url = [], [], []

for file_name, rels in final_result.items():
    doc_url = basename_to_url.get(file_name)
    if not doc_url:
        missing_url.append(file_name)
        continue
    for desc, parsed in rels.items():
        reasoning = parsed.get("reasoning", "")
        if "lower_domain" in parsed:
            lower_rows.append({"docURL": doc_url, "description": desc,
                               "domain": parsed["lower_domain"],
                               "reasoning": reasoning})
        elif "mid_domain" in parsed:
            mid_rows.append({"docURL": doc_url, "description": desc,
                             "domain": parsed["mid_domain"],
                             "reasoning": reasoning})

print(f"lower rows: {len(lower_rows)} | mid rows: {len(mid_rows)} | "
      f"files w/o docURL: {len(missing_url)}")
if missing_url:
    print("  unmatched files:", missing_url[:5])


lower rows: 6070 | mid rows: 707 | files w/o docURL: 0


In [36]:
# ── 2. Cypher (relation match scoped to the document) ──
LOWER_CYPHER = f"""
UNWIND $rows AS row
MATCH (d:Document {{docURL: row.docURL}})<-[:PART_OF]-(:Chunk)-[:ASSERTS]
      ->(r:Relation {{relationshipDescription: row.description}})
MATCH (d)-[:HAS_LOW_LEVEL]->(l:LowLevelDomain {{{DOMAIN_PROP}: row.domain}})
MERGE (r)-[c:CLASSIFIED_AS]->(l)
SET c.reasoning = row.reasoning
RETURN count(*) AS linked
"""

MID_CYPHER = f"""
UNWIND $rows AS row
MATCH (d:Document {{docURL: row.docURL}})<-[:PART_OF]-(:Chunk)-[:ASSERTS]
      ->(r:Relation {{relationshipDescription: row.description}})
MATCH (d)-[:HAS_MID_LEVEL]->(m:MidLevelDomain {{{DOMAIN_PROP}: row.domain}})
MERGE (r)-[c:CLASSIFIED_AS]->(m)
SET c.reasoning = row.reasoning
RETURN count(*) AS linked
"""

# ── 3. Run in batches ──
def run_in_batches(session, cypher, rows, batch=500):
    total = 0
    for i in range(0, len(rows), batch):
        rec = session.run(cypher, rows=rows[i:i + batch]).single()
        total += rec["linked"] if rec else 0
    return total

with neo4j_driver.session() as session:
    n_low = run_in_batches(session, LOWER_CYPHER, lower_rows)
    n_mid = run_in_batches(session, MID_CYPHER, mid_rows)

print(f"Linked {n_low} lower-domain edges and {n_mid} mid-domain edges "
      f"(out of {len(lower_rows) + len(mid_rows)} classified)")


Linked 3344 lower-domain edges and 697 mid-domain edges (out of 6777 classified)


In [37]:
non_general = [r for r in lower_rows + mid_rows if r["domain"] != "GENERAL"]
print(f"non-GENERAL classified: {len(non_general)} | linked: 3207")


non-GENERAL classified: 4010 | linked: 3207
